# 배경이 합쳐진 글자 이미지 보정

정사각형 2×2 입력 한 장과 칸별 정답 기준 획을 사용합니다. 입력은 576×576으로 맞춥니다. 짙은 글자와 상대적으로 밝은 배경에서 시험하세요.
**4번에서 기준 글자 순서 지정 → 5번에서 완성 이미지 업로드 → 6번 모델 로딩 → 7번 삭제·추가 마스크 확인 → 8번 실행 → 9번 ZIP.**
7번에서 어둡고 채도가 낮은 잉크를 흰 배경에 남겨 획을 추출합니다. 빨강은 원본에서 인페인팅할 곳, 파랑은 잉크를 추가할 곳입니다.
8번에서 흰 배경 위의 획을 보정한 뒤, 지운 자리에 복원한 배경과 보정한 잉크를 합쳐 컬러 결과를 만듭니다.
가려진 배경은 주변을 보고 추정합니다. 글자 위치는 2×2의 각 칸으로 지정합니다.


In [ ]:
# 1. 설치 및 고정 런타임 준비. 이 노트북만으로 실행 가능
import sys, subprocess, json, hashlib, gc, shutil, traceback, time, zipfile
from pathlib import Path
from datetime import datetime
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.37.0', 'transformers==4.57.6', 'accelerate', 'safetensors',
    'timm', 'numpy', 'scipy', 'scikit-image', 'pillow', 'pandas', 'tqdm', 'ipywidgets'], check=True)
RUNTIME_HELPERS={'stroke_model/reference_model.py': '"""Reference-conditioned V2. Inputs: white ink on black, Bx1xHxW in [0,1].\n\nNo thresholding, CPU conversion or detached image features in the forward path.\nOld checkpoints are deliberately not compatible with this new architecture.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef block(cin, cout, stride=1):\n    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=stride, padding=1),\n                         nn.GroupNorm(4, cout), nn.GELU())\n\n\ndef resize(x, size):\n    return F.interpolate(x, size=size, mode="bilinear", align_corners=False)\n\n\ndef warp(strokes, flow):\n    b, _, h, w = strokes.shape\n    ys = (torch.arange(h, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / h - 1\n    xs = (torch.arange(w, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / w - 1\n    y, x = torch.meshgrid(ys, xs, indexing="ij")\n    grid = torch.stack((x, y), -1)[None].expand(b, -1, -1, -1)\n    offset = torch.stack((flow[:, 0] * 2 / w, flow[:, 1] * 2 / h), -1)\n    return F.grid_sample(strokes, grid + offset, align_corners=False)\n\n\nclass ReferenceStrokeModel(nn.Module):\n    def __init__(self, image_size=128, width=64, pretrained=True,\n                 backbone=None, channels=None, channels_last=False, stroke_chunk_size=4):\n        super().__init__()\n        if width % 4:\n            raise ValueError("width must be divisible by 4")\n        mean, std = (.485, .456, .406), (.229, .224, .225)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(\n                "swin_tiny_patch4_window7_224", pretrained=pretrained,\n                features_only=True, out_indices=(0, 1, 2, 3),\n                in_chans=3, img_size=image_size, drop_path_rate=0.)\n            channels = backbone.feature_info.channels()\n            channels_last = True  # timm Swin features are NHWC.\n            mean = backbone.pretrained_cfg.get("mean", mean)\n            std = backbone.pretrained_cfg.get("std", std)\n        if not channels:\n            raise ValueError("Injected backbone requires its feature channels")\n        self.backbone = backbone\n        self.stroke_chunk_size = max(1, int(stroke_chunk_size))\n        self.channels_last = channels_last\n        self.backbone_frozen = True\n        self.backbone.requires_grad_(False)\n        self.backbone.eval()\n        self.register_buffer("rgb_mean", torch.tensor(mean).view(1, 3, 1, 1))\n        self.register_buffer("rgb_std", torch.tensor(std).view(1, 3, 1, 1))\n        self.lateral = nn.ModuleList(nn.Conv2d(c, width, 1) for c in channels)\n        self.smooth = nn.ModuleList(block(width, width) for _ in channels)\n        self.detail = block(1, width)  # Full-resolution branch for thin strokes.\n        self.fuse = block(width * 2, width)\n        self.flow_net = nn.Sequential(block(2, 16, 2), block(16, 32, 2),\n                                      nn.Conv2d(32, 2, 3, padding=1))\n        nn.init.zeros_(self.flow_net[-1].weight)\n        nn.init.zeros_(self.flow_net[-1].bias)\n        # Shared across stroke IDs; full glyph + selected stroke + xy context.\n        self.reference = nn.Sequential(block(4, 16, 2), block(16, width, 2))\n        self.head = nn.Sequential(block(width * 2 + 4, width),\n                                  nn.Conv2d(width, 1, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.backbone_frozen:\n            self.backbone.eval()\n        return self\n\n    def forward(self, image, reference, strokes, active=None):\n        if image.ndim != 4 or image.shape[1] != 1 or reference.shape != image.shape:\n            raise ValueError("image/reference must have matching Bx1xHxW shapes")\n        b, _, h, w = image.shape\n        if strokes.ndim != 4 or strokes.shape[0] != b or strokes.shape[-2:] != (h, w):\n            raise ValueError("strokes must have shape BxKxHxW")\n        k = strokes.shape[1]\n        if active is None:\n            active = strokes.flatten(2).sum(-1) > 0  # Fixed reference metadata only.\n        if active.shape != (b, k) or active.dtype != torch.bool or not active.any(1).all():\n            raise ValueError("Each sample needs at least one active reference stroke")\n        rgb = (1 - image).expand(-1, 3, -1, -1)\n        feats = self.backbone((rgb - self.rgb_mean) / self.rgb_std)\n        if self.channels_last:\n            feats = [f.permute(0, 3, 1, 2) for f in feats]\n        pyramid = None\n        for i in reversed(range(len(feats))):\n            lateral = self.lateral[i](feats[i])\n            pyramid = lateral if pyramid is None else lateral + resize(pyramid, lateral.shape[-2:])\n            pyramid = self.smooth[i](pyramid)\n        features = self.fuse(torch.cat((resize(pyramid, (h, w)), self.detail(image)), 1))\n        flow = 12 * resize(self.flow_net(torch.cat((reference, image), 1)), (h, w)).tanh()\n        hints = warp(strokes, flow)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=image.device, dtype=image.dtype),\n                                torch.linspace(-1, 1, w, device=image.device, dtype=image.dtype), indexing="ij")\n        xy = torch.stack((xx, yy))[None].expand(b, -1, -1, -1)\n        logits = self.score_strokes(features, reference, strokes, xy, flow, hints, image)\n        logits = logits.float().masked_fill(~active[:, :, None, None], -torch.inf)\n        probabilities = logits.softmax(1) * active[:, :, None, None]\n        probabilities = probabilities / probabilities.sum(1, keepdim=True)\n        return {"logits": logits, "probabilities": probabilities,\n                "masks": image * probabilities, "flow": flow, "hints": hints,\n                "active": active}\n\n    def score_strokes(self, features, reference, strokes, xy, flow, hints, image):\n        """Same head algebra, batching strokes and computing shared convolution once.\n\n        GroupNorm is per sample, so stroke batching does not mix stroke statistics.\n        Existing V2 state_dicts remain compatible; input gradients remain connected.\n        """\n        b, k, h, w = strokes.shape\n        width = features.shape[1]\n        conv = self.head[0][0]\n        shared_weight = torch.cat((conv.weight[:, :width], conv.weight[:, 2*width+1:]), 1)\n        shared = F.conv2d(torch.cat((features, image, xy), 1), shared_weight, conv.bias, padding=1)\n        outputs = []\n        for start in range(0, k, self.stroke_chunk_size):\n            n = min(self.stroke_chunk_size, k-start)\n            def repeat(t):\n                return t[:, None].expand(-1, n, -1, -1, -1).reshape(b*n, *t.shape[1:])\n            selected = strokes[:, start:start+n].reshape(b*n, 1, h, w)\n            r = self.reference(torch.cat((repeat(reference), selected, repeat(xy)), 1))\n            r = warp(resize(r, (h, w)), repeat(flow))\n            condition = torch.cat((r, hints[:, start:start+n].reshape(b*n, 1, h, w)), 1)\n            x = F.conv2d(condition, conv.weight[:, width:2*width+1], padding=1) + repeat(shared)\n            x = self.head[0][2](self.head[0][1](x))\n            outputs.append(self.head[1](x).reshape(b, n, h, w))\n        return torch.cat(outputs, 1)\n\n\ndef exclusive_masks(output, foreground):\n    """Export only: non-differentiable exact partition of the supplied foreground."""\n    expected = output[\'probabilities\'][:, :1].shape\n    if foreground.shape != expected:\n        raise ValueError(\'foreground must have shape Bx1xHxW\')\n    labels = output["probabilities"].argmax(1)\n    masks = F.one_hot(labels, output["probabilities"].shape[1]).permute(0, 3, 1, 2)\n    return masks.bool() & foreground.bool()\n\n\ndef stroke_moments(masks, eps=1e-6):\n    """Area fraction, centroid x/y and xx/xy/yy covariance in normalized coordinates.\n\n    Near-empty strokes have unreliable geometry: callers must mask/inspect them.\n    """\n    h, w = masks.shape[-2:]\n    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=masks.device, dtype=masks.dtype),\n                          torch.linspace(-1, 1, w, device=masks.device, dtype=masks.dtype), indexing="ij")\n    mass = masks.sum((-2, -1))\n    weights = masks / mass.clamp_min(eps)[..., None, None]\n    cx, cy = (weights * x).sum((-2, -1)), (weights * y).sum((-2, -1))\n    dx, dy = x - cx[..., None, None], y - cy[..., None, None]\n    return torch.stack((mass / (h * w), cx, cy,\n                        (weights * dx.square()).sum((-2, -1)),\n                        (weights * dx * dy).sum((-2, -1)),\n                        (weights * dy.square()).sum((-2, -1))), -1)\n\n\ndef partition_loss(output, labels):\n    """labels: BxHxW, 0..K-1 for owned pixels; -1 background/unknown.\n\n    Unknown crossing pixels are excluded, not assigned pseudo-certainty.\n    Soft Dice averages only annotated, nonempty target strokes.\n    """\n    logits = output["logits"]\n    valid = labels >= 0\n    if labels.shape != logits.shape[:1] + logits.shape[2:]:\n        raise ValueError("labels must have shape BxHxW")\n    if not valid.any() or (labels < -1).any() or (labels >= logits.shape[1]).any():\n        raise ValueError("Need annotated pixels with labels in -1..K-1")\n    target = F.one_hot(labels.clamp_min(0).long(), logits.shape[1]).permute(0, 3, 1, 2)\n    target = target * valid[:, None]\n    if (target.bool() & ~output["active"][:, :, None, None]).any():\n        raise ValueError("Target refers to an inactive reference stroke")\n    ce = F.cross_entropy(logits, labels.long(), ignore_index=-1)\n    pred = output["probabilities"] * valid[:, None]\n    mass = target.sum((-2, -1))\n    dice = (2 * (pred * target).sum((-2, -1)) + 1e-6) / (pred.sum((-2, -1)) + mass + 1e-6)\n    dice_loss = (1 - dice)[mass > 0].mean()\n    flow = output["flow"]\n    smooth = (flow[..., 1:, :] - flow[..., :-1, :]).square().mean()\n    smooth = smooth + (flow[..., :, 1:] - flow[..., :, :-1]).square().mean()\n    return {"loss": ce + dice_loss + .001 * smooth, "ce": ce,\n            "dice_loss": dice_loss, "flow_smoothness": smooth}\n', 'stroke_model/stroke_structure.py': '"""Reference-aware experimental stroke losses; inputs are absolute ink in [0, 1].\n\nReference preparation is discrete and cached. All prediction-dependent losses use\nPyTorch operations, including min/max subgradients. No predicted components are\ndiscarded. Scores are uncalibrated errors, not human-quality probabilities.\n"""\nfrom dataclasses import dataclass, fields\nimport math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, label\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass StructureConfig:\n    # All distances are pixels at the inference resolution (normally 128).\n    reference_threshold: float = 0.5\n    coverage_radius: int = 2\n    required_ink: float = 0.7\n    anchor_spacing: int = 10\n    anchor_radius: int = 1\n    corridor_radius: int = 3\n    pair_spans: tuple = (1, 2)\n    max_pairs: int = 32\n    connectivity_steps: int | None = None  # None: crop area, with convergence check\n    skeleton_iterations: int = 12\n    junction_radii: tuple = (1, 2, 3)\n    junction_margin: float = 0.0  # No font/short-branch exemption by default\n    junction_thresholds: tuple = (0.15, 0.30, 0.45, 0.60, 0.75)\n    junction_temperature: float = 0.025  # Soft occupancy transition, absolute ink\n    junction_arm_scale: float = 1.25\n    junction_arm_offset: float = 1.0\n    junction_min_arm_length: float = 2.0\n    junction_merge_radius: int = 1\n    junction_match_radius: int = 2\n    junction_min_levels: int = 2\n    junction_probe_scale: float = 1.5\n    junction_probe_offset: float = 1.0\n    junction_distance_steps: int = 12\n    junction_angular_samples: int = 48\n    junction_angular_smoothing: int = 3\n    junction_ring_margin: float = 0.1\n    junction_seed_radius: int = 2\n    attachment_steps: int | None = None  # None: H*W, with convergence check\n    expected_width: float = 5.0  # Optional normal-width prior, NOT skeleton width\n    width_radius: int = 4\n    width_tolerance: float = 0.25\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    width_weight: float = 0.0  # Enable only after normal-width calibration\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name in (\'coverage_radius\', \'anchor_radius\', \'corridor_radius\',\n                     \'junction_seed_radius\', \'width_radius\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        for name in (\'anchor_spacing\', \'max_pairs\', \'skeleton_iterations\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 1:\n                raise ValueError(f\'{name} must be a positive integer\')\n        if self.anchor_spacing <= 2 * self.anchor_radius + 1:\n            raise ValueError(\'anchor_spacing must exceed the anchor diameters\')\n        if not self.pair_spans or any(not isinstance(v, int) or v < 1 for v in self.pair_spans):\n            raise ValueError(\'pair_spans must contain positive integers\')\n        if not self.junction_radii or any(not isinstance(v, int) or v < 1 for v in self.junction_radii):\n            raise ValueError(\'junction_radii must contain integers >= 1\')\n        if (not self.junction_thresholds or any(not math.isfinite(float(v)) or not 0 < v < 1\n                                               for v in self.junction_thresholds)\n                or sorted(set(self.junction_thresholds)) != list(self.junction_thresholds)):\n            raise ValueError(\'junction_thresholds must be strictly increasing values in (0,1)\')\n        if not math.isfinite(self.junction_temperature) or self.junction_temperature <= 0:\n            raise ValueError(\'junction_temperature must be finite and positive\')\n        for name in (\'junction_arm_scale\', \'junction_min_arm_length\', \'junction_probe_scale\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0:\n                raise ValueError(f\'{name} must be positive\')\n        for name in (\'junction_arm_offset\', \'junction_probe_offset\', \'junction_ring_margin\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be nonnegative\')\n        for name in (\'junction_merge_radius\', \'junction_match_radius\', \'junction_angular_smoothing\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        if (not isinstance(self.junction_min_levels, int)\n                or not 1 <= self.junction_min_levels <= len(self.junction_thresholds)):\n            raise ValueError(\'junction_min_levels must fit the threshold count\')\n        if not isinstance(self.junction_distance_steps, int) or self.junction_distance_steps < 1:\n            raise ValueError(\'junction_distance_steps must be a positive integer\')\n        if not isinstance(self.junction_angular_samples, int) or self.junction_angular_samples < 16:\n            raise ValueError(\'junction_angular_samples must be an integer >= 16\')\n        for name in (\'connectivity_steps\', \'attachment_steps\'):\n            v = getattr(self, name)\n            if v is not None and (not isinstance(v, int) or v < 1):\n                raise ValueError(f\'{name} must be None or a positive integer\')\n        if not 0 < self.required_ink <= 1 or not 0 < self.reference_threshold < 1:\n            raise ValueError(\'required_ink must be in (0,1]; reference_threshold in (0,1)\')\n        if self.expected_width <= 0 or self.eps <= 0 or not 0 <= self.width_tolerance < 1:\n            raise ValueError(\'Invalid width or eps\')\n        if self.junction_margin < 0 or any(getattr(self, n) < 0 for n in\n                (\'missing_weight\', \'split_weight\', \'junction_weight\', \'width_weight\')):\n            raise ValueError(\'Weights and junction_margin must be nonnegative\')\n        if any(not math.isfinite(float(getattr(self, f.name))) for f in fields(self)\n               if isinstance(getattr(self, f.name), (float, int))):\n            raise ValueError(\'Configuration values must be finite\')\n\n\ndef _pool(x, radius, mean=False):\n    k = 2 * radius + 1\n    return (F.avg_pool2d(x, k, 1, radius) if mean else F.max_pool2d(x, k, 1, radius))\n\n\ndef soft_skeleton(x, iterations):\n    """Pooling skeleton from soft-clDice; a surrogate, not exact graph thinning."""\n    def erode(v):\n        return torch.minimum(-F.max_pool2d(-v, (3, 1), 1, (1, 0)),\n                             -F.max_pool2d(-v, (1, 3), 1, (0, 1)))\n    def opening(v):\n        return F.max_pool2d(erode(v), 3, 1, 1)\n    skel = F.relu(x - opening(x))\n    for _ in range(iterations):\n        x = erode(x)\n        delta = F.relu(x - opening(x))\n        skel = skel + F.relu(delta - skel * delta)\n    return skel\n\n\ndef reconstruct(mask, seed, steps=None):\n    """8-connected grayscale reconstruction; no dilation/closing of mask.\n\n    Returns (reachable strength, converged). None allows H*W iterations, a\n    conservative finite-grid bound. Early stopping reads detached values only;\n    the returned tensor retains the graph. Explicit small step caps can truncate.\n    """\n    reached = torch.minimum(mask, seed)\n    limit = mask.shape[-2] * mask.shape[-1] if steps is None else steps\n    converged = False\n    for _ in range(limit):\n        updated = torch.minimum(mask, F.max_pool2d(reached, 3, 1, 1))\n        if torch.equal(updated.detach(), reached.detach()):\n            reached = updated\n            converged = True\n            break\n        reached = updated\n    return reached, converged\n\n\ndef _ordered_centerline(binary):\n    skel = skeletonize(binary)\n    _, count = label(skel, structure=np.ones((3, 3)))\n    if count != 1:\n        raise ValueError(\'Reference stroke must have one connected centerline; clean the reference\')\n    points = [tuple(v) for v in np.argwhere(skel)]\n    lookup = set(points)\n    neighbors = {}\n    for y, x in points:\n        adjacent = []\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                q = (y + dy, x + dx)\n                if (not dy and not dx) or q not in lookup:\n                    continue\n                # Avoid triangular shortcut edges at a normal digital corner.\n                if dy and dx and ((y + dy, x) in lookup or (y, x + dx) in lookup):\n                    continue\n                adjacent.append(q)\n        neighbors[(y, x)] = adjacent\n    if any(len(v) > 2 for v in neighbors.values()):\n        raise ValueError(\'Reference skeleton branches; provide a clean unbranched reference stroke\')\n    endpoints = sorted(p for p, n in neighbors.items() if len(n) == 1)\n    closed = len(points) > 2 and not endpoints\n    if len(points) < 5 or (not closed and len(endpoints) != 2):\n        raise ValueError(\'Reference centerline must be a nontrivial open curve or closed ring\')\n    start = endpoints[0] if endpoints else min(points)\n    order, visited, current = [], set(), start\n    while current not in visited:\n        order.append(current)\n        visited.add(current)\n        choices = sorted(q for q in neighbors[current] if q not in visited)\n        if not choices:\n            break\n        current = choices[0]\n    if len(order) != len(points):\n        raise ValueError(\'Cannot order reference centerline without losing pixels\')\n    return skel, np.asarray(order), closed\n\n\ndef _disk_region(shape, point, radius):\n    yy, xx = np.indices(shape)\n    return ((yy - point[0]) ** 2 + (xx - point[1]) ** 2 <= radius ** 2)\n\n\nclass _Pair(nn.Module):\n    def __init__(self, corridor, a, b, slices):\n        super().__init__()\n        for name, value in [(\'corridor\', corridor), (\'a\', a), (\'b\', b)]:\n            self.register_buffer(name, torch.tensor(value, dtype=torch.float32)[None, None])\n        self.slices = slices\n\n\nclass _Stroke(nn.Module):\n    def __init__(self, reference, config):\n        super().__init__()\n        skel, order, self.closed = _ordered_centerline(reference > config.reference_threshold)\n        self.length = len(order)\n        self.register_buffer(\'skeleton\', torch.tensor(skel, dtype=torch.float32)[None, None])\n        self.register_buffer(\'ys\', torch.tensor(order[:, 0], dtype=torch.long))\n        self.register_buffer(\'xs\', torch.tensor(order[:, 1], dtype=torch.long))\n        distance = distance_transform_edt(~skel)\n        self.register_buffer(\'seeds\', torch.tensor(distance <= config.junction_seed_radius,\n                                                   dtype=torch.float32)[None, None])\n        normal = distance <= max(0.5, config.expected_width / 2)\n        self.register_buffer(\'normal\', torch.tensor(normal, dtype=torch.float32)[None, None])\n        anchors = list(range(0, len(order), config.anchor_spacing))\n        if not self.closed and anchors[-1] != len(order) - 1:\n            anchors.append(len(order) - 1)\n        raw_pairs = []\n        for span in config.pair_spans:\n            for i in range(len(anchors)):\n                j = i + span\n                if not self.closed and j >= len(anchors):\n                    continue\n                if self.closed and span >= len(anchors):\n                    continue\n                begin, end = anchors[i], anchors[j % len(anchors)]\n                path = (order[begin:end + 1] if end > begin else\n                        np.concatenate((order[begin:], order[:end + 1])))\n                if len(path) <= 2 * config.anchor_radius + 2:\n                    continue\n                # Keep anchor patches disjoint, including on tight curves.\n                if np.linalg.norm(order[begin] - order[end]) <= 2 * config.anchor_radius + 1:\n                    continue\n                segment = np.zeros_like(skel)\n                segment[path[:, 0], path[:, 1]] = True\n                corridor = distance_transform_edt(~segment) <= config.corridor_radius\n                a = _disk_region(skel.shape, order[begin], config.anchor_radius) & corridor\n                b = _disk_region(skel.shape, order[end], config.anchor_radius) & corridor\n                yy, xx = np.where(corridor)\n                slices = (slice(yy.min(), yy.max() + 1), slice(xx.min(), xx.max() + 1))\n                raw_pairs.append((corridor[slices], a[slices], b[slices], slices))\n        if len(raw_pairs) > config.max_pairs:\n            ids = np.linspace(0, len(raw_pairs) - 1, config.max_pairs).round().astype(int)\n            raw_pairs = [raw_pairs[i] for i in ids]\n        self.pairs = nn.ModuleList([_Pair(*args) for args in raw_pairs])\n        if not self.pairs:\n            raise ValueError(\'No valid anchor pairs: reduce anchor_spacing/radius for this reference\')\n\n\ndef _ring_offsets(radius):\n    # Sample a digital square perimeter in cyclic order, with no duplicates.\n    return ([( -radius, x) for x in range(-radius, radius)] +\n            [(y, radius) for y in range(-radius, radius)] +\n            [(radius, x) for x in range(radius, -radius, -1)] +\n            [(y, -radius) for y in range(radius, -radius, -1)])\n\n\ndef _crossings(skeleton, radius):\n    h, w = skeleton.shape[-2:]\n    padded = F.pad(skeleton, (radius,) * 4)\n    samples = [padded[..., radius + y:radius + y + h, radius + x:radius + x + w]\n               for y, x in _ring_offsets(radius)]\n    return sum((a - b).abs() for a, b in zip(samples, samples[1:] + samples[:1])) / 2\n\n\ndef soft_occupancy(mask, threshold, temperature):\n    """Approximate absolute level-set occupancy, with exactly 0->0 and 1->1.\n\n    This is NOT per-stroke max normalization. An empty/faint missing stroke is\n    never amplified relative to its own maximum. Shape and absolute ink are\n    separated using shared, configurable thresholds.\n    """\n    tau = torch.as_tensor(threshold, device=mask.device, dtype=mask.dtype)\n    low = torch.sigmoid(-tau / temperature)\n    high = torch.sigmoid((1 - tau) / temperature)\n    return ((torch.sigmoid((mask - tau) / temperature) - low) / (high - low)).clamp(0, 1)\n\n\ndef soft_distance_to_background(occupancy, steps):\n    """Differentiable truncated chamfer distance, in input pixels.\n\n    Fixed iterations; min has the same subgradient qualification as pooling.\n    This estimates the probe scale; it never edits or fills the input mask.\n    """\n    distance = occupancy * steps\n    h, w = occupancy.shape[-2:]\n    for _ in range(steps):\n        padded = F.pad(distance, (1, 1, 1, 1), value=0)\n        neighbors = [distance]\n        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)):\n            neighbors.append(padded[..., 1+dy:1+dy+h, 1+dx:1+dx+w] + math.hypot(dy, dx))\n        distance = torch.stack(neighbors).amin(0)\n    return distance\n\n\ndef occupancy_ring_arms(occupancy, radius, samples, smoothing):\n    """Count resolved angular transitions on the actual occupancy, not skeleton."""\n    n, _, h, w = occupancy.shape\n    angles = torch.arange(samples, device=occupancy.device, dtype=occupancy.dtype) * (2*math.pi/samples)\n    yy, xx = torch.meshgrid(torch.arange(h, device=occupancy.device, dtype=occupancy.dtype),\n                            torch.arange(w, device=occupancy.device, dtype=occupancy.dtype), indexing=\'ij\')\n    x = xx[None, :, :, None] + radius[:, 0, :, :, None] * angles.cos()\n    y = yy[None, :, :, None] + radius[:, 0, :, :, None] * angles.sin()\n    grid = torch.stack(((x+.5)*2/w-1, (y+.5)*2/h-1), -1).reshape(n, h, w*samples, 2)\n    ring = F.grid_sample(occupancy, grid, align_corners=False).reshape(n, 1, h, w, samples)\n    for _ in range(smoothing):\n        ring = (ring.roll(1, -1) + 2*ring + ring.roll(-1, -1))/4\n    return (ring-ring.roll(1, -1)).abs().sum(-1)/2\n\n\ndef multilevel_junction(mask, seeds, config):\n    """Differentiate shape per absolute level BEFORE counting skeleton arms.\n\n    Levels are stacked on the batch axis. Return a strongest-level response map\n    and level diagnostics. A branch with ink .6 is not counted as 3*.6 arms.\n    """\n    b, _, h, w = mask.shape\n    levels = len(config.junction_thresholds)\n    tau = mask.new_tensor(config.junction_thresholds)[:, None, None, None, None]\n    occupancy = soft_occupancy(mask[None], tau, config.junction_temperature).reshape(levels*b, 1, h, w)\n    skeleton = soft_skeleton(occupancy, config.skeleton_iterations)\n    attached, converged = reconstruct(occupancy, occupancy * seeds, config.attachment_steps)\n    local = torch.stack([F.relu(_crossings(skeleton, r)-2-config.junction_margin).square()\n                         for r in config.junction_radii]).mean(0)\n    local = F.max_pool2d(local, 5, 1, 2)\n    distance = soft_distance_to_background(occupancy, config.junction_distance_steps)\n    radius = config.junction_probe_scale*distance + config.junction_probe_offset\n    arms = occupancy_ring_arms(occupancy, radius, config.junction_angular_samples,\n                               config.junction_angular_smoothing)\n    resolved = F.relu(arms-2-config.junction_ring_margin).square()\n    maps = (resolved * local * skeleton * attached).reshape(levels, b, 1, h, w)\n    # Average evidence instead of selecting a worst pixel from any one threshold.\n    response = maps.mean(0)\n    return response, maps, converged\n\n\nclass StrokeStructureLoss(nn.Module):\n    """Prepare once on fixed aligned reference strokes, then call on live masks.\n\n    reference_strokes: K,H,W. forward: K,H,W or B,K,H,W, floating [0,1].\n    Returned per_stroke values are B,K. loss is an EXTRACTOR structural loss,\n    not a source-image quality score. Reference deficits are returned separately.\n    Closed curves are ordered\n    cyclically; local corridors prevent bypassing a gap via the rest of a ring.\n    No alignment is estimated here. Geometry must already be in input coordinates.\n    """\n    def __init__(self, reference_strokes, config=None):\n        super().__init__()\n        self.config = config or StructureConfig()\n        ref = torch.as_tensor(reference_strokes).detach().float().cpu().numpy()\n        if ref.ndim != 3 or not np.isfinite(ref).all() or (ref < 0).any() or (ref > 1).any():\n            raise ValueError(\'reference_strokes must be finite K,H,W in [0,1]\')\n        self.shape = ref.shape\n        prepared = []\n        for k, stroke in enumerate(ref):\n            try:\n                prepared.append(_Stroke(stroke, self.config))\n            except ValueError as exc:\n                raise ValueError(f\'Reference stroke {k + 1}: {exc}\') from exc\n        if not prepared:\n            raise ValueError(\'At least one reference stroke is required\')\n        self.strokes = nn.ModuleList(prepared)\n\n    def forward(self, masks, return_maps=False):\n        if masks.ndim == 3:\n            masks = masks[None]\n        if masks.ndim != 4 or tuple(masks.shape[1:]) != tuple(self.shape):\n            raise ValueError(f\'masks must have trailing shape {self.shape}\')\n        if not masks.is_floating_point():\n            raise ValueError(\'masks must be floating point\')\n        if not bool(torch.isfinite(masks).all()) or bool((masks < 0).any()) or bool((masks > 1).any()):\n            raise ValueError(\'masks must be finite in [0,1]; do not normalize each stroke\')\n        if next(self.buffers()).device != masks.device:\n            raise ValueError(\'Move the prepared loss to masks.device with .to(device)\')\n        cfg = self.config\n        metrics = {name: [] for name in (\'missing\', \'split\', \'junction\', \'width\')}\n        all_maps, convergence = [], []\n        for k, stroke in enumerate(self.strokes):\n            m = masks[:, k:k + 1].float()\n            coverage = _pool(m, cfg.coverage_radius)[:, 0, stroke.ys, stroke.xs]\n            deficits = F.relu(1 - coverage / cfg.required_ink).square()\n            missing = deficits.mean(-1)\n            local_mass = _pool(m, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            expected = _pool(stroke.normal, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            width = F.relu(1 - local_mass / (expected * (1 - cfg.width_tolerance) + cfg.eps)).square().mean(-1)\n            split_values, split_maps, pair_converged = [], [], []\n            for pair in stroke.pairs:\n                f = m[..., pair.slices[0], pair.slices[1]] * pair.corridor\n                reached, converged = reconstruct(f, f * pair.a, cfg.connectivity_steps)\n                u = (f * pair.a).flatten(1).amax(1)\n                v = (f * pair.b).flatten(1).amax(1)\n                c = (reached * pair.b).flatten(1).amax(1)\n                strength = torch.minimum(u, v)\n                # clamp avoids the tiny nonzero epsilon penalty on perfect paths.\n                gap = F.relu((strength - c) / strength.clamp_min(cfg.eps)).square()\n                split_values.append(u * v * gap)\n                pair_converged.append(converged)\n                if return_maps:\n                    split_maps.append((pair, F.relu(f - reached)))\n            split = torch.stack(split_values, -1).mean(-1)\n            attached, attach_converged = reconstruct(m, m * stroke.seeds, cfg.attachment_steps)\n            junction_map, junction_levels, junction_converged = multilevel_junction(m, stroke.seeds, cfg)\n            junction = junction_map.flatten(1).sum(-1) / stroke.length\n            for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction), (\'width\', width)]:\n                metrics[name].append(value)\n            convergence.append({\'stroke\': k + 1, \'pairs\': all(pair_converged),\n                                \'attachment\': attach_converged and junction_converged, \'closed\': stroke.closed})\n            if return_maps:\n                missing_map = torch.zeros_like(m)\n                missing_map[:, 0, stroke.ys, stroke.xs] = deficits\n                split_map = torch.zeros_like(m)\n                for pair, area in split_maps:\n                    # Maps are diagnostics, not the differentiable loss path.\n                    with torch.no_grad():\n                        target = split_map[..., pair.slices[0], pair.slices[1]]\n                        target.copy_(torch.maximum(target, area.detach()))\n                all_maps.append({\'mask\': m.detach(), \'centerline\': stroke.skeleton.detach(),\n                                 \'missing\': missing_map.detach(), \'split\': split_map.detach(),\n                                 \'junction\': junction_map.detach(), \'attached\': attached.detach(),\n                                 \'junction_level_scores\': (junction_levels.flatten(2).sum(-1) / stroke.length).detach()})\n        per_stroke = {key: torch.stack(value, -1) for key, value in metrics.items()}\n        extraction = cfg.split_weight * per_stroke[\'split\'] + cfg.junction_weight * per_stroke[\'junction\']\n        reference = cfg.missing_weight * per_stroke[\'missing\'] + cfg.width_weight * per_stroke[\'width\']\n        result = {\'loss\': extraction.mean(), \'reference_loss\': reference.mean(),\n                  \'per_stroke\': {**per_stroke, \'extraction_error\': extraction,\n                                 \'reference_deficit\': reference},\n                  \'source_quality\': {\'assessed\': False,\n                      \'reason\': \'Extracted-mask defects do not identify source-image defects.\'},\n                  \'convergence\': convergence}\n        if return_maps:\n            result[\'maps\'] = all_maps\n        return result\n', 'stroke_model/source_structure.py': '"""Source-image errors after ONE bounded affine registration of the target.\n\nExtractor channel mistakes are diagnostics, not evidence of missing source ink.\nRegistration is CPU evaluation preprocessing, not differentiable guidance.\nPreparedSourceMetric retains image gradients with the registration held fixed.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport numpy as np\nfrom scipy.ndimage import affine_transform, distance_transform_edt\nfrom scipy.optimize import minimize\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom .stroke_structure import (StructureConfig, _Stroke, _pool, reconstruct,\n    soft_occupancy, soft_skeleton, soft_distance_to_background, occupancy_ring_arms)\n\n\n@dataclass(frozen=True)\nclass SourceConfig:\n    translation: float = 12.\n    rotation: float = 15.\n    scale_min: float = .85\n    scale_max: float = 1.2\n    shear: float = .2\n    registration_iterations: int = 45\n    registration_prior: float = .00001\n    extra_radius: float = 2.\n    extra_transition: float = .5\n    junction_tolerance: float = .5\n    junction_position_tolerance: int = 2\n    coverage_radius: int = 2\n    corridor_radius: int = 3\n\n    def __post_init__(self):\n        if any(not math.isfinite(x) for x in asdict(self).values()):\n            raise ValueError(\'Source settings must be finite\')\n        if (self.translation <= 0 or self.rotation < 0 or self.shear < 0\n                or not 0 < self.scale_min <= 1 <= self.scale_max or self.scale_min == self.scale_max\n                or self.registration_iterations < 1 or self.registration_prior < 0\n                or self.extra_radius < 0 or self.extra_transition <= 0 or self.junction_tolerance < 0):\n            raise ValueError(\'Invalid source settings\')\n        for key in (\'registration_iterations\',\'junction_position_tolerance\',\'coverage_radius\',\'corridor_radius\'):\n            if not isinstance(getattr(self,key), int) or getattr(self,key)<0:\n                raise ValueError(\'Source iteration counts and radii must be nonnegative integers\')\n\n\ndef affine_warp(array, parameters):\n    tx, ty, angle, sx, sy, shear = parameters\n    angle = np.deg2rad(angle)\n    rotation = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])\n    matrix = rotation @ np.array([[sy, shear], [0, sx]])  # y,x convention\n    inverse = np.linalg.inv(matrix)\n    center = (np.asarray(array.shape)-1)/2\n    offset = center-inverse@(center+np.array([ty,tx]))\n    return affine_transform(array, inverse, offset, order=1, mode=\'constant\')\n\n\ndef register_reference(image, reference, strokes, config=None):\n    cfg = config or SourceConfig()\n    image, reference, strokes = [np.asarray(x, dtype=np.float32) for x in (image,reference,strokes)]\n    image, reference = image.squeeze(), reference.squeeze()\n    if (image.ndim!=2 or reference.shape!=image.shape or strokes.ndim!=3\n            or strokes.shape[1:]!=image.shape or not len(strokes)):\n        raise ValueError(\'Need matching input H,W, full target H,W and target strokes K,H,W\')\n    for value in (image,reference,strokes):\n        if not np.isfinite(value).all() or (value<0).any() or (value>1).any():\n            raise ValueError(\'Source arrays must be finite absolute ink in [0,1]\')\n    if reference.max()<=.5 or any(s.max()<=.5 for s in strokes):\n        raise ValueError(\'Target and all expected target strokes must be nonempty\')\n    def objective(parameters):\n        moved = affine_warp(reference, parameters)\n        dice = 2*(image*moved).sum()/(np.square(image).sum()+np.square(moved).sum()+1e-6)\n        return 1-dice+cfg.registration_prior*sum(v*v for v in parameters[:3])\n    bounds=[(-cfg.translation,cfg.translation),(-cfg.translation,cfg.translation),\n            (-cfg.rotation,cfg.rotation),(cfg.scale_min,cfg.scale_max),\n            (cfg.scale_min,cfg.scale_max),(-cfg.shear,cfg.shear)]\n    fit=minimize(objective,[0,0,0,1,1,0],method=\'Powell\',bounds=bounds,\n                 options={\'maxiter\':cfg.registration_iterations,\'xtol\':.001,\'ftol\':.00001})\n    # An optimizer result may be worse than identity; preserve the best of both.\n    parameters = fit.x if objective(fit.x)<objective([0,0,0,1,1,0]) else np.array([0,0,0,1,1,0])\n    target=affine_warp(reference,parameters)\n    dice=2*(image*target).sum()/(np.square(image).sum()+np.square(target).sum()+1e-6)\n    # Keep reference centerline topology intact; sample the live input into\n    # reference coordinates instead of re-skeletonizing a raster-warped target.\n    return reference, strokes, {\'parameters\':parameters.tolist(),\'dice\':float(dice),\n                            \'optimizer_converged\':bool(fit.success),\'fixed_for_gradients\':True}\n\n\nclass PreparedSourceMetric(nn.Module):\n    def __init__(self, aligned_target, aligned_strokes, config=None, parameters=None):\n        super().__init__()\n        self.config=config or SourceConfig()\n        self.structure_config=StructureConfig(coverage_radius=self.config.coverage_radius,\n            corridor_radius=self.config.corridor_radius, reference_threshold=.5)\n        self.strokes=nn.ModuleList([_Stroke(s,self.structure_config) for s in aligned_strokes])\n        self.register_buffer(\'target\',torch.tensor(aligned_target,dtype=torch.float32)[None,None])\n        distance=distance_transform_edt(np.asarray(aligned_target)<.3)\n        support=1/(1+np.exp(np.clip((distance-self.config.extra_radius)/self.config.extra_transition,-80,80)))\n        # Exact support=1 inside target ink; avoid charging even an identical\n        # image for the sigmoid\'s nonzero interior tail.\n        support=np.minimum(support/(1/(1+np.exp(-self.config.extra_radius/self.config.extra_transition))),1)\n        self.register_buffer(\'support\',torch.tensor(support,dtype=torch.float32)[None,None])\n        pose=parameters if parameters is not None else [0,0,0,1,1,0]\n        source_support=affine_warp(support,pose)\n        self.register_buffer(\'source_support\',torch.tensor(source_support,dtype=torch.float32)[None,None])\n        weights=np.stack([np.exp(-distance_transform_edt(~(s>.15))/8.) for s in aligned_strokes])\n        self.register_buffer(\'junction_weights\',torch.tensor(weights,dtype=torch.float32))\n        h,w=self.target.shape[-2:]\n        tx,ty,angle,sx,sy,shear=parameters if parameters is not None else [0,0,0,1,1,0]\n        angle=np.deg2rad(angle)\n        rotation=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])\n        matrix=rotation@np.array([[sy,shear],[0,sx]])\n        yy,xx=np.indices((h,w));center=(np.array([h,w])-1)/2\n        positions=matrix@np.stack([yy-center[0],xx-center[1]]).reshape(2,-1)\n        positions+= (center+np.array([ty,tx]))[:,None]\n        grid=np.stack([(positions[1].reshape(h,w)+.5)*2/w-1,\n                       (positions[0].reshape(h,w)+.5)*2/h-1],axis=-1)\n        self.register_buffer(\'sampling_grid\',torch.tensor(grid,dtype=torch.float32)[None])\n\n    def forward(self,image):\n        image=torch.as_tensor(image)\n        if image.ndim==2:image=image[None,None]\n        elif image.ndim==3:image=image[None]\n        if image.ndim!=4 or image.shape[1:]!=self.target.shape[1:]:\n            raise ValueError(\'Image must be H,W, 1,H,W or B,1,H,W matching the prepared target\')\n        if not image.is_floating_point():image=image.float()\n        if image.device!=self.target.device or not bool(torch.isfinite(image).all()) or bool(((image<0)|(image>1)).any()):\n            raise ValueError(\'Image must be finite ink [0,1] on the metric device\')\n        source_image=image\n        image=F.grid_sample(image,self.sampling_grid.to(image).expand(image.shape[0],-1,-1,-1),align_corners=False)\n        cfg=self.structure_config\n        missing,split=[],[]\n        for stroke in self.strokes:\n            coverage=_pool(image,cfg.coverage_radius)[:,0,stroke.ys,stroke.xs]\n            missing.append(F.relu(1-coverage/cfg.required_ink).square().mean(-1))\n            pairs=[]\n            for pair in stroke.pairs:\n                ink=image[...,pair.slices[0],pair.slices[1]]*pair.corridor\n                reached,_=reconstruct(ink,ink*pair.a)\n                u=(ink*pair.a).flatten(1).amax(1)\n                v=(ink*pair.b).flatten(1).amax(1)\n                c=(reached*pair.b).flatten(1).amax(1)\n                pairs.append(u*v*F.relu((torch.minimum(u,v)-c)/torch.minimum(u,v).clamp_min(cfg.eps)).square())\n            split.append(torch.stack(pairs,-1).mean(-1))\n        levels=[]\n        for threshold in cfg.junction_thresholds:\n            occupancy=soft_occupancy(image,threshold,cfg.junction_temperature)\n            expected=soft_occupancy(self.target.to(image),threshold,cfg.junction_temperature).expand_as(image)\n            radius=cfg.junction_probe_scale*soft_distance_to_background(occupancy,cfg.junction_distance_steps)+cfg.junction_probe_offset\n            arms=occupancy_ring_arms(occupancy,radius,cfg.junction_angular_samples,cfg.junction_angular_smoothing)\n            target_arms=occupancy_ring_arms(expected,radius,cfg.junction_angular_samples,cfg.junction_angular_smoothing)\n            allowed=_pool(target_arms,self.config.junction_position_tolerance).clamp_min(2)\n            excess=F.relu(arms-allowed-self.config.junction_tolerance).square()\n            levels.append(excess*soft_skeleton(occupancy,cfg.skeleton_iterations))\n        junction_map=torch.stack(levels).mean(0)\n        junction=torch.stack([(junction_map[:,0]*w).flatten(1).sum(1)/s.length\n                              for w,s in zip(self.junction_weights,self.strokes)],-1)\n        # Inspect EVERY original pixel, including margins outside the registered\n        # canonical view; an affine crop must never hide an added stroke.\n        extra_map=source_image*(1-self.source_support)\n        extra=extra_map.flatten(1).sum(1)/source_image.flatten(1).sum(1).clamp_min(1)\n        return {\'missing\':torch.stack(missing,-1),\'split\':torch.stack(split,-1),\'junction\':junction,\n                \'extra\':extra,\'extra_map\':extra_map,\'junction_map\':junction_map,\'support\':self.source_support,\n                \'registered_image\':image,\'source_image\':source_image}\n', 'stroke_model/target_stroke_plan.py': '"""Observed widths -> canonical strokes -> one glyph-level affine -> edit plan.\n\nScores select edit strength; they are not differentiated through a generator.\nThe built-in geometry scores are explicit uncalibrated engineering proxies, NOT\na reproduction of Li & Ren\'s calligraphy evaluator or extraction confidence.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import distance_transform_edt, binary_dilation, convolve\nfrom skimage.morphology import skeletonize\nimport torch\n\nfrom .source_structure import SourceConfig, register_reference, affine_warp\n\n\n@dataclass(frozen=True)\nclass TargetPlanConfig:\n    mask_threshold: float = .35\n    keep_score: float = 90.\n    full_edit_score: float = 40.\n    maximum_step_blend: float = .8\n    edit_margin: int = 2\n    width_min_samples: int = 6\n\n    def __post_init__(self):\n        if any(not math.isfinite(v) for v in asdict(self).values()):raise ValueError(\'Nonfinite config\')\n        if not 0<self.mask_threshold<1 or not 0<=self.full_edit_score<self.keep_score<=100:\n            raise ValueError(\'Invalid threshold or score range\')\n        if not 0<self.maximum_step_blend<=1 or type(self.edit_margin) is not int or self.edit_margin<0:\n            raise ValueError(\'Invalid blend/margin\')\n        if type(self.width_min_samples) is not int or self.width_min_samples<1:raise ValueError(\'Invalid sample count\')\n\n\ndef _features(mask):\n    binary=mask>.35;path=skeletonize(binary)\n    if not path.any():return dict(present=False)\n    # Undirected 8-neighbour edge length, counting each edge once.\n    length=0.\n    for dy,dx,cost in [(0,1,1.),(1,0,1.),(1,1,2**.5),(1,-1,2**.5)]:\n        a=path[:path.shape[0]-dy or None, max(0,-dx):path.shape[1]-max(0,dx)]\n        b=path[dy:,max(0,dx):path.shape[1]-max(0,-dx)]\n        length+=float((a&b).sum())*cost\n    xy=np.argwhere(path)[:,::-1].astype(float)\n    center=xy.mean(0);cov=np.cov(xy.T) if len(xy)>2 else np.eye(2)\n    values,vectors=np.linalg.eigh(cov);axis=vectors[:,-1]\n    ambiguous=bool(values[-1]<1.15*max(values[0],1e-6))\n    angle=float(np.arctan2(axis[1],axis[0]))\n    # Bend is a dimensionless path/span proxy, not circle-fit curvature.\n    span=max(float(np.linalg.norm(xy.max(0)-xy.min(0))),1.)\n    return dict(present=True,length=length,width=float(2*np.median(distance_transform_edt(binary)[path])),\n        angle=angle,angle_ambiguous=ambiguous,bend=length/span,\n        center=center.tolist(),area=int(binary.sum()))\n\n\ndef evaluate_stroke(observed,target):\n    a,b=_features(observed),_features(target)\n    if not a[\'present\']:\n        return dict(score=0.,source=\'geometry_proxy_v1\',observed=a,target=b,reason=\'absent_observed_stroke\')\n    length_ratio=1. if a[\'length\']<1 and b[\'length\']<1 else a[\'length\']/max(b[\'length\'],1.)\n    width_ratio=a[\'width\']/max(b[\'width\'],1e-6)\n    angle_delta=float((a[\'angle\']-b[\'angle\']+np.pi/2)%np.pi-np.pi/2)\n    angle_used=not (a[\'angle_ambiguous\'] or b[\'angle_ambiguous\'])\n    bend_error=abs(a[\'bend\']-b[\'bend\'])\n    ma,mb=observed>.35,target>.35\n    dice=float(2*(ma&mb).sum()/max(int(ma.sum()+mb.sum()),1))\n    # Width is measured and reported, but NOT forced back to template width.\n    error=(abs(math.log(max(length_ratio,1e-6)))+\n           (abs(angle_delta)/(np.pi/4) if angle_used else 0.)+.5*bend_error+(1-dice))\n    return dict(score=float(100*np.exp(-error)),source=\'geometry_proxy_v1\',observed=a,target=b,\n        length_ratio=length_ratio,width_ratio=width_ratio,angle_delta_radians=angle_delta,\n        angle_used=angle_used,bend_error=bend_error,body_dice=dice)\n\n\ndef measure_widths(observed,config):\n    estimates=[]\n    for mask in observed:\n        binary=mask>config.mask_threshold;path=skeletonize(binary)\n        degree=convolve(path.astype(int),np.ones((3,3),int),mode=\'constant\')-path\n        stable=path&(degree==2)\n        values=distance_transform_edt(binary)[stable]\n        estimate=float(np.median(values)) if len(values)>=config.width_min_samples else None\n        estimates.append(estimate)\n    reliable=[w for w in estimates if w is not None and w>0]\n    if not reliable:raise ValueError(\'No measurable observed stroke widths; defer this glyph\')\n    median=float(np.median(reliable))\n    return [dict(half_width=w if w is not None else median,\n                 source=\'observed_stroke\' if w is not None else \'observed_glyph_median\') for w in estimates]\n\n\ndef _background(rgb,foreground):\n    if foreground.all():raise ValueError(\'No background pixels available\')\n    _,indices=distance_transform_edt(foreground,return_indices=True)\n    return rgb[indices[0],indices[1]]\n\n\nclass TargetImageGuide:\n    def __init__(self,source,target,weights,stroke_regions,report,config):\n        self.source=np.asarray(source,np.float32);self.target=np.asarray(target,np.float32)\n        self.weights=np.asarray(weights,np.float32)\n        self.stroke_regions=stroke_regions;self.report=report;self.config=config\n        self.roi=self.weights>0\n        self.metadata=dict(kind=\'observed_width_aligned_target_image\',config=asdict(config),\n            score_role=\'frozen edit selection and blend strength; not a scalar reward gradient\',\n            final_pixel_replacement=False)\n\n    @torch.no_grad()\n    def project(self,rgb):\n        if rgb.shape!=(1,3,*self.weights.shape):raise ValueError(\'Expected matching RGB sheet\')\n        target=torch.as_tensor(self.target.copy(),device=rgb.device).permute(2,0,1)[None].to(rgb)\n        weight=torch.as_tensor(self.weights.copy(),device=rgb.device)[None,None].to(rgb)\n        return rgb+weight*(target-rgb)\n\n    @torch.no_grad()\n    def diagnostics(self,rgb):\n        target=torch.as_tensor(self.target.copy(),device=rgb.device).permute(2,0,1)[None].to(rgb)\n        error=(rgb-target).square().mean(1)[0]\n        rows=[]\n        for region,item in zip(self.stroke_regions,self.report[\'strokes\']):\n            mask=torch.as_tensor(region.copy(),device=rgb.device)\n            rows.append(dict(glyph=item[\'glyph\'],stroke=item[\'stroke\'],original_score=item[\'score\'],\n                edit_strength=item[\'edit_strength\'],target_rgb_mse=float(error[mask].mean()) if mask.any() else 0.))\n        return dict(strokes=rows,target_rgb_mse=float(error.mean()),correctness_score=False)\n\n\ndef build_target_plan(source,conditions,observed_masks,*,scores=None,config=None,registration_config=None):\n    """Build a full target IMAGE before deriving local, score-controlled edits.\n\n    Optional scores: four lists of 0..100 consistency scores, in stroke-ID order.\n    Omit scores to use the disclosed geometry proxy. Correct extraction/canonical\n    stroke correspondence is assumed; extraction reliability is not predicted.\n    """\n    cfg=config or TargetPlanConfig();source=np.asarray(source.convert(\'RGB\'),np.float32)/255\n    h,w=source.shape[:2]\n    if len(conditions)!=4 or len(observed_masks)!=4 or h!=w or h%2:raise ValueError(\'Expected a 2x2 sheet\')\n    if scores is not None and len(scores)!=4:raise ValueError(\'Need four per-glyph score lists\')\n    target=source.copy();weights=np.zeros((h,w),np.float32);regions=[];rows=[];glyph_reports=[];rendered=[]\n    for gi,(condition,observed) in enumerate(zip(conditions,observed_masks)):\n        strokes=np.asarray(condition[\'strokes\'],np.float32);observed=np.asarray(observed,np.float32)\n        if observed.shape!=strokes.shape or strokes.ndim!=3 or not len(strokes):raise ValueError(\'Stroke masks must correspond\')\n        if any(not np.isfinite(a).all() or a.min()<0 or a.max()>1 for a in (strokes,observed)):\n            raise ValueError(\'Expected finite ink masks in [0,1]\')\n        size=strokes.shape[-2:];side=w//2;x0=(gi%2)*side;y0=(gi//2)*side\n        crop=source[y0:y0+side,x0:x0+side];luma=crop@np.array([.299,.587,.114])\n        widths=measure_widths(observed,cfg)\n        masks=[]\n        for stroke,width in zip(strokes,widths):\n            path=skeletonize(stroke>.35)\n            if not path.any():raise ValueError(\'Empty canonical stroke\')\n            # Render EVERY skeleton, including closed curves, at measured width.\n            masks.append(np.clip(width[\'half_width\']+.5-distance_transform_edt(~path),0,1))\n        masks=np.asarray(masks,np.float32)\n        source_ink=np.asarray(Image.fromarray((1-luma).astype(np.float32)).resize(size[::-1],Image.Resampling.BILINEAR))\n        _,_,registration=register_reference(source_ink,masks.max(0),masks,registration_config or SourceConfig())\n        aligned=np.stack([affine_warp(mask,registration[\'parameters\']) for mask in masks]).clip(0,1)\n        supplied=None if scores is None else scores[gi]\n        if supplied is not None and (len(supplied)!=len(strokes) or any(not math.isfinite(s) or not 0<=s<=100 for s in supplied)):\n            raise ValueError(\'Scores must match stroke IDs and be in [0,100]\')\n        foreground=(1-luma)>.35\n        background=_background(crop,foreground)\n        patch=background.copy()\n        glyph_weights=np.zeros(size,np.float32)\n        for k,(alpha,actual,width) in enumerate(zip(aligned,observed,widths)):\n            evaluation=evaluate_stroke(actual,alpha)\n            score=evaluation[\'score\'] if supplied is None else float(supplied[k])\n            strength=float(np.clip((cfg.keep_score-score)/(cfg.keep_score-cfg.full_edit_score),0,1))\n            region=(alpha>.03)|(actual>.15)\n            if cfg.edit_margin:region=binary_dilation(region,iterations=cfg.edit_margin)\n            glyph_weights=np.maximum(glyph_weights,region*strength*cfg.maximum_step_blend)\n            resized=np.asarray(Image.fromarray(alpha).resize((side,side),Image.Resampling.BICUBIC)).clip(0,1)\n            observation=np.asarray(Image.fromarray(actual).resize((side,side),Image.Resampling.BILINEAR))\n            samples=crop[observation>.5]\n            if not len(samples):samples=crop[foreground]\n            color=np.median(samples,axis=0) if len(samples) else np.zeros(3)\n            patch=patch*(1-resized[...,None])+color*resized[...,None]\n            full=np.zeros((h,w),bool)\n            full[y0:y0+side,x0:x0+side]=np.asarray(Image.fromarray(region).resize((side,side),Image.Resampling.NEAREST))\n            regions.append(full)\n            rows.append(dict(glyph=gi+1,stroke=k+1,score=score,\n                score_source=evaluation[\'source\'] if supplied is None else \'provided_per_stroke_score\',\n                edit_strength=strength,width=width,measurements=evaluation))\n        target[y0:y0+side,x0:x0+side]=patch\n        weights[y0:y0+side,x0:x0+side]=np.asarray(Image.fromarray(glyph_weights).resize((side,side),Image.Resampling.BILINEAR))\n        glyph_reports.append(dict(glyph=gi+1,bbox=[x0,y0,x0+side,y0+side],registration=registration))\n        rendered.append(dict(thick_strokes=masks,aligned_strokes=aligned))\n    report=dict(strokes=rows,glyphs=glyph_reports,target_built_from=\'canonical skeletons + observed widths, then shared glyph affine\',\n        score_calibration=\'uncalibrated geometry proxy unless supplied; not paper reproduction\',\n        width_used_for=\'appearance preservation, not forced to reference thickness\',\n        limitations=[\'Canonical stroke correspondence is assumed correct.\',\'Geometry proxy is not a correctness probability.\',\n            \'Nearest visible background is only a provisional target background, not scene inpainting.\',\n            \'Registration may fail on severe errors; inspect parameters and Dice before interpreting results.\'])\n    return TargetImageGuide(source,target,weights,regions,report,cfg),rendered\n', 'diffusion_model/stroke_projected_diffusion.py': '"""Paired DDIM sampling with direct per-stroke x0 projection, no backward pass.\n\nSupply a loaded SD1.5 img2img pipeline with DDIM scheduler and float32 VAE.\nThis is a sampling experiment, not a trained glyph-condition model.\n"""\nfrom copy import deepcopy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\n\ndef run_projected_pair(pipe, source, constraints, output, *, prompt=\'\', negative_prompt=\'\',\n                       steps=60,strength=.25,seed=2026,cfg_scale=5.,start_from_noise=False,\n                       save_final_projection=False,progress=None,guided_only=False,save_step_images=False):\n    from PIL import Image\n    out=Path(output);out.mkdir(parents=True,exist_ok=False)\n    if not 0<strength<=1 or steps<1 or (not start_from_noise and int(steps*strength)<1) or any(v%8 for v in source.size):\n        raise ValueError(\'Invalid schedule or image size\')\n    if type(pipe.scheduler).__name__ != \'DDIMScheduler\':\n        raise ValueError(\'This experimental sampler supports DDIM only\')\n    device=next(pipe.unet.parameters()).device\n    rgb=torch.tensor(np.asarray(source.convert(\'RGB\')).copy(),device=device).permute(2,0,1)[None].float()/255\n    constraints.diagnostics(rgb)\n    def encode(x):return pipe.vae.encode(x.float()*2-1).latent_dist.mode()*pipe.vae.config.scaling_factor\n    def decode(z):return (pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,return_dict=False)[0].float()/2+.5).clamp(0,1)\n    def save(x,path):\n        Image.fromarray(np.rint(x[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(\'uint8\')).save(path)\n    traces={}\n    step_images=[]\n    finish=getattr(constraints,\'finish\',None)\n    if save_step_images: (out/\'steps\').mkdir()\n    with torch.no_grad():\n        positive,negative=pipe.encode_prompt(prompt,device,1,True,negative_prompt=negative_prompt)\n        embeddings=torch.cat([negative,positive])\n        pipe.scheduler.set_timesteps(steps,device=device)\n        if start_from_noise:\n            ts=pipe.scheduler.timesteps\n        else:\n            ts,_=pipe.get_timesteps(steps,strength,device)\n        clean=encode(rgb)\n        noise=torch.randn(clean.shape,generator=torch.Generator(\'cpu\').manual_seed(seed)).to(device)\n        initial=noise*pipe.scheduler.init_noise_sigma if start_from_noise else pipe.scheduler.add_noise(clean,noise,ts[:1])\n        roi=torch.as_tensor(constraints.roi.copy(),device=device)[None,None].float()\n        gate=F.adaptive_max_pool2d(roi,initial.shape[-2:])\n        arms=(\'stroke_projected\',) if guided_only else (\'redenoise_only\',\'stroke_projected\')\n        for arm in arms:\n            scheduler=deepcopy(pipe.scheduler);latent=initial.clone();trace=[]\n            for index,t in enumerate(ts):\n                model_input=scheduler.scale_model_input(torch.cat([latent,latent]),t)\n                u,c=pipe.unet(model_input.to(next(pipe.unet.parameters()).dtype),t,\n                    encoder_hidden_states=embeddings,return_dict=False)[0].chunk(2)\n                base=scheduler.step((u+cfg_scale*(c-u)).float(),t,latent,eta=0.,return_dict=True)\n                latent=base.prev_sample.detach();row=dict(step=index,timestep=int(t))\n                if arm==\'stroke_projected\':\n                    prediction=decode(base.pred_original_sample)\n                    target=constraints.project(prediction)\n                    # Re-encoding is not an exact pixel projection. Record both.\n                    # In ink mode subtract the unedited RGB round-trip so only\n                    # the requested edit contributes to the latent correction.\n                    reference_latent=encode(prediction) if finish is not None else base.pred_original_sample\n                    corrected=base.pred_original_sample+gate*(encode(target)-reference_latent)\n                    previous=int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                    alpha=(scheduler.alphas_cumprod[previous] if previous>=0 else scheduler.final_alpha_cumprod).to(latent)\n                    latent=latent+alpha.sqrt()*(corrected-base.pred_original_sample)\n                    corrected_rgb=decode(corrected)\n                    row.update(before=constraints.diagnostics(prediction),pixel_projected=constraints.diagnostics(target),\n                               after_vae=constraints.diagnostics(corrected_rgb))\n                    if save_step_images:\n                        frame=f\'steps/{index+1:03d}.png\'\n                        save(corrected_rgb,out/frame)\n                        step_images.append(dict(step=index+1,timestep=int(t),path=frame))\n                trace.append(row)\n                if progress:progress(arm,index+1,len(ts),row)\n            final=decode(latent)\n            if arm==\'stroke_projected\' and finish is not None:\n                save(final,out/\'stroke_projected_raw.png\')\n                final=finish(final)\n            save(final,out/(arm+\'.png\'))\n            if arm==\'stroke_projected\' and save_final_projection:\n                # Explicitly separate the raw diffusion output from the final\n                # image-space enforcement. Never label this as raw generation.\n                save(constraints.project(final),out/\'stroke_projected_final_pixel_projection.png\')\n            traces[arm]=trace\n    source.save(out/\'input.png\')\n    report=dict(steps=steps,actual_steps=len(ts),strength=None if start_from_noise else strength,\n        start_from_noise=start_from_noise,source_content_used=not start_from_noise,seed=seed,cfg_scale=cfg_scale,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=not guided_only,backward_passes=0,additional_training=False,traces=traces,\n        step_images=step_images,\n        guide=getattr(constraints,\'metadata\',dict(kind=\'per_stroke_ink_constraints\')),\n        final_pixel_projection_saved=save_final_projection,\n        saved_png_ink_tolerance=1/255,\n        limitations=[\'Target construction and extraction correspondence must be checked.\',\n                     \'VAE re-encoding can change the target and leak changes beyond a pixel mask.\',\n                     \'A separately requested final pixel projection is postprocessing, not raw generation.\'])\n    (out/\'report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    return report\n', 'repair_audit/stage1.py': '"""First-stage diagnosis of the existing four-glyph cyclic repair, without training.\n\nReads data (never runtime code) from existing sweep/cyclic ZIPs. Automatic width\nand reference-distance measurements are proxies, not character correctness.\n"""\nfrom copy import deepcopy\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import binary_dilation, distance_transform_edt, maximum_filter\nfrom skimage.morphology import skeletonize\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef json_write(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef _rgb(z, name):\n    with Image.open(io.BytesIO(z.read(name))) as image:\n        return image.convert(\'RGB\').copy()\n\n\ndef _conditions(z, group):\n    result = []\n    for slot in range(1, 5):\n        member = f\'{group}/source_conditions/condition_{slot}.npz\'\n        with np.load(io.BytesIO(z.read(member)), allow_pickle=False) as arrays:\n            reference = arrays[\'reference\'].astype(np.float32)\n            strokes = arrays[\'strokes\'].astype(np.float32)\n        if (reference.ndim != 3 or reference.shape[0] != 1 or\n                reference.shape[1] != reference.shape[2] or strokes.ndim != 3 or\n                strokes.shape[1:] != reference.shape[1:] or not len(strokes)):\n            raise ValueError(f\'Invalid reference/stroke dimensions: {member}\')\n        for value in (reference, strokes):\n            if not np.isfinite(value).all() or value.min() < 0 or value.max() > 1:\n                raise ValueError(f\'Expected finite [0,1] ink arrays: {member}\')\n        if reference.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n            raise ValueError(f\'Empty glyph/stroke: {member}\')\n        result.append(dict(reference=reference, strokes=strokes))\n    return result\n\n\ndef read_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Materialize only named PNG/NPZ data, ignoring saved absolute paths/code."""\n    with zipfile.ZipFile(path) as check:\n        if \'archive_metadata.json\' in check.namelist():\n            return read_stage1_archive(path, output, selected_case_ids, include_controls)\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    selected = None if selected_case_ids is None else set(selected_case_ids)\n    with zipfile.ZipFile(path) as z:\n        experiment = json.loads(z.read(\'experiment.json\'))\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Archive experiment.json must record the extractor SHA256\')\n        cases = json.loads(z.read(\'cases.json\'))\n        available = {c[\'case_id\'] for c in cases if not c.get(\'control\')}\n        if selected is not None and (not selected or selected - available):\n            raise ValueError(f\'Unknown/empty case selection; available: {sorted(available)}\')\n        chosen = [c for c in cases if not c.get(\'control\') and\n                  (selected is None or c[\'case_id\'] in selected)]\n        if not chosen:\n            raise ValueError(\'No generated cases in this archive\')\n        prepared, controls = [], set()\n        destination.mkdir(parents=True)\n        for item in chosen:\n            group, case_id = item[\'group_id\'], item[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+\', group) or not re.fullmatch(r\'[A-Za-z0-9_-]+\', case_id):\n                raise ValueError(\'Invalid archive group/case identifier\')\n            conditions = _conditions(z, group)\n            report = json.loads(z.read(f\'{group}/generation/generation_report.json\'))\n            basename = PurePosixPath(str(item[\'input\']).replace(\'\\\\\', \'/\')).name\n            matches = [p for p in report[\'pages\'] if\n                       PurePosixPath(p[\'input\']).name == basename and not p.get(\'control\')]\n            if len(matches) != 1:\n                raise ValueError(f\'Generated page is not uniquely recorded: {case_id}\')\n            pages = [(case_id, matches[0], \'generated\')]\n            if include_controls and group not in controls:\n                candidates = [p for p in report[\'pages\'] if p.get(\'control\') and p[\'glyph_count\'] == 4]\n                if len(candidates) != 1:\n                    raise ValueError(\'Need one recorded four-glyph normal input control\')\n                pages.append((group + \'_normal_control\', candidates[0], \'normal_control\'))\n                controls.add(group)\n            for identifier, page, kind in pages:\n                if page[\'glyph_count\'] != 4 or page.get(\'safety_filtered\'):\n                    raise ValueError(\'Requires a non-filtered four-glyph page\')\n                member = f"{group}/generation/page_{page[\'page_id\']:03d}.png"\n                source = _rgb(z, member)\n                if source.width != source.height or source.width % 16:\n                    raise ValueError(\'Expected square four-cell image with side divisible by 16\')\n                target = destination / identifier\n                target.mkdir()\n                source.save(target / \'input.png\')\n                for slot, condition in enumerate(conditions, 1):\n                    np.savez_compressed(target / f\'condition_{slot}.npz\', **condition)\n                metadata = dict(case_id=identifier, kind=kind, archive_member=member,\n                    source_sha256=hashlib.sha256(z.read(member)).hexdigest(),\n                    prompt=report.get(\'prompts\', {}).get(\'4\', item.get(\'prompt\', \'\')),\n                    negative_prompt=report.get(\'negative_prompt\', \'\'),\n                    generation_seed=page.get(\'seed\'), generation_strength=page.get(\'strength\'),\n                    checkpoint_sha256=expected.lower())\n                json_write(target / \'input_metadata.json\', metadata)\n                prepared.append(dict(directory=str(target), **metadata))\n    json_write(destination / \'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared))\n    return prepared, experiment\n\n\ndef read_stage1_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Reuse original inputs from an audit ZIP, never the already guided outputs."""\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(path) as z:\n        metadata = json.loads(z.read(\'archive_metadata.json\'))\n        experiment = metadata[\'experiment\']\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Missing original extractor SHA256\')\n        recorded = metadata[\'selected_cases\']\n        available = {c[\'case_id\'] for c in recorded if c[\'kind\'] == \'generated\'}\n        selected = available if selected_case_ids is None else set(selected_case_ids)\n        if not selected or selected - available:\n            raise ValueError(f\'Unknown/empty audit selection; available: {sorted(available)}\')\n        chosen = [c for c in recorded if c[\'case_id\'] in selected]\n        groups = {re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\']).group(1)\n                  for c in chosen if re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\'])}\n        if len(groups) == 0:\n            raise ValueError(\'Unsupported audit case identifiers\')\n        if include_controls:\n            controls = {c[\'case_id\']: c for c in recorded if c[\'kind\'] == \'normal_control\'}\n            for group in sorted(groups):\n                name = group + \'_normal_control\'\n                if name not in controls:\n                    raise ValueError(\'Audit ZIP lacks the matching normal control\')\n                chosen.append(controls[name])\n        prepared = []\n        destination.mkdir(parents=True)\n        for entry in chosen:\n            identifier = entry[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+_(page_[0-9]+|normal_control)\', identifier):\n                raise ValueError(\'Invalid audit identifier\')\n            target = destination / identifier\n            target.mkdir()\n            image = _rgb(z, identifier + \'/input.png\')\n            if image.width != image.height or image.width % 16:\n                raise ValueError(\'Invalid input sheet dimensions\')\n            image.save(target / \'input.png\')\n            # Same validation as the original four source conditions.\n            for slot in range(1, 5):\n                with np.load(io.BytesIO(z.read(f\'{identifier}/condition_{slot}.npz\')), allow_pickle=False) as a:\n                    ref, strokes = a[\'reference\'].astype(np.float32), a[\'strokes\'].astype(np.float32)\n                if (ref.ndim != 3 or ref.shape[0] != 1 or ref.shape[1] != ref.shape[2] or\n                        strokes.ndim != 3 or not len(strokes) or strokes.shape[1:] != ref.shape[1:]):\n                    raise ValueError(\'Invalid condition shape\')\n                if any(not np.isfinite(v).all() or v.min() < 0 or v.max() > 1 for v in (ref, strokes)):\n                    raise ValueError(\'Invalid condition values\')\n                if ref.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n                    raise ValueError(\'Empty condition\')\n                np.savez_compressed(target/f\'condition_{slot}.npz\', reference=ref, strokes=strokes)\n            row = dict(entry, directory=str(target), checkpoint_sha256=expected.lower(),\n                       reused_stage1_original=True)\n            json_write(target/\'input_metadata.json\', row)\n            prepared.append(row)\n    json_write(destination/\'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared, original_input_reused=True))\n    return prepared, experiment\n\n\ndef load_conditions(directory):\n    result = []\n    for slot in range(1, 5):\n        with np.load(Path(directory) / f\'condition_{slot}.npz\', allow_pickle=False) as a:\n            result.append(dict(reference=a[\'reference\'].copy(), strokes=a[\'strokes\'].copy()))\n    return result\n\n\ndef fit_records(source, conditions, extractor, output, *, fit=True):\n    """Same fitting parameters as the existing cyclic notebook."""\n    import torch\n    from torch.nn import functional as F\n    from stroke_model.stroke_reference_fit import fit_reference_glyph, ReferenceFitConfig\n    from diffusion_model.fitted_candidate_diffusion import image_tensor\n    directory = Path(output)\n    directory.mkdir(parents=True, exist_ok=True)\n    device = next(extractor.parameters()).device\n    rgb = image_tensor(source, device)\n    ink = 1 - (rgb * rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n    side = source.width // 2\n    records = []\n    for i, condition in enumerate(conditions):\n        ref, strokes = condition[\'reference\'], condition[\'strokes\']\n        x, y = (i % 2) * side, (i // 2) * side\n        crop = F.interpolate(ink[..., y:y+side, x:x+side], ref.shape[-2:], mode=\'bilinear\', align_corners=False)\n        with torch.no_grad():\n            prediction = extractor(crop, torch.as_tensor(ref, device=device)[None],\n                                   torch.as_tensor(strokes, device=device)[None])\n        masks = prediction[\'masks\'][0].cpu().numpy()\n        if not fit:\n            np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n                extracted_masks=masks, probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n            continue\n        hints = prediction.get(\'hints\', torch.as_tensor(strokes, device=device)[None])[0].cpu().numpy()\n        fitted = fit_reference_glyph(strokes, masks, hints=hints, glyph_reference=ref,\n            config=ReferenceFitConfig(max_displacement=16., smoothness=12., shape_prior=.15, max_junction_shift=2.))\n        record = dict(reference=ref, strokes=strokes, fitted=fitted[\'strokes\'], extracted_masks=masks,\n                      fit_metadata=[r[\'metadata\'] for r in fitted[\'fits\']])\n        records.append(record)\n        np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n            extracted_masks=masks, fitted_strokes=fitted[\'strokes\'],\n            probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n        json_write(directory / f\'slot_{i+1}_fit.json\', record[\'fit_metadata\'])\n    return records\n\n\ndef ink_array(image):\n    a = np.asarray(image, np.float32) / 255\n    return 1 - (a * np.array([.299, .587, .114], np.float32)).sum(-1)\n\n\ndef fixed_support(source, controller):\n    """Freeze source-based diagnostic support before comparing ANY output arm."""\n    initial_edit = np.maximum(controller.erase.cpu().numpy()[0, 0], controller.restore.cpu().numpy()[0, 0]) > .01\n    preserve = ~binary_dilation(initial_edit, iterations=3)\n    source_ink = ink_array(source)\n    healthy = (source_ink > .5) & preserve\n    center = skeletonize(source_ink > .5) & healthy\n    return dict(healthy=healthy, center=center, preserve=preserve, initial_edit=initial_edit)\n\n\ndef measure(source, result, reference, support, threshold=.5):\n    """Fixed-coordinate width proxy; movement/thresholds confound it. No OCR claim."""\n    original, output, ref = map(ink_array, (source, result, reference))\n    healthy, center = support[\'healthy\'], support[\'center\']\n    source_radius = distance_transform_edt(original > threshold)[center]\n    output_radius = distance_transform_edt(output > threshold)[center]\n    ratio = float(output_radius.mean() / max(source_radius.mean(), 1e-8)) if len(source_radius) else None\n    union = (original > .2) | (ref > .2)\n    def average(values, mask):\n        return float(values[mask].mean()) if mask.any() else None\n    return dict(threshold=threshold, fixed_center_samples=int(center.sum()),\n        healthy_ink_pixels=int(healthy.sum()),\n        fixed_center_width_ratio_proxy=ratio,\n        healthy_ink_drop=average(np.maximum(original-output, 0), healthy),\n        healthy_ink_retention=average((output > threshold).astype(float), healthy),\n        healthy_ink_mse=average((original-output)**2, healthy),\n        reference_distance_on_fixed_union=average((output-ref)**2, union),\n        source_reference_distance_on_fixed_union=average((original-ref)**2, union))\n\n\ndef reference_sheet(conditions, size):\n    result = Image.new(\'RGB\', (size, size), \'white\')\n    half = size // 2\n    for i, c in enumerate(conditions):\n        glyph = Image.fromarray(np.rint((1-c[\'reference\'][0])*255).astype(np.uint8)).convert(\'RGB\')\n        result.paste(glyph.resize((half, half), Image.Resampling.BICUBIC), ((i % 2)*half, (i//2)*half))\n    return result\n\n\ndef width_probe_conditions(conditions):\n    """Optional +1 model-pixel dilation stress test, NOT a validated new font."""\n    return [dict(reference=maximum_filter(c[\'reference\'], size=(1, 3, 3)),\n                 strokes=maximum_filter(c[\'strokes\'], size=(1, 3, 3))) for c in conditions]\n\n\ndef run_case(pipe, extractor, case, *, steps=60, strength=.25, seed=2026,\n             cfg_scale=5., width_probe=False, progress=None,\n             boundary_guard=False, boundary_config=None, observed_width=False, width_config=None,\n             resume=False, on_arm_complete=None):\n    import torch\n    from diffusion_model.fitted_candidate_diffusion import image_tensor, optimize_clean\n    from stroke_model.restore_settle_erase import RestoreSettleErase\n    if steps < 4 or not 0 < strength < 1 or int(steps*strength) < 4:\n        raise ValueError(\'Need at least four actual repair steps\')\n    directory = Path(case[\'directory\'])\n    if observed_width and not boundary_guard:\n        raise ValueError(\'Observed-width comparison requires boundary_guard=True\')\n    out = directory / \'audit\'\n    out.mkdir(exist_ok=resume)\n    with Image.open(directory / \'input.png\') as im:\n        source = im.convert(\'RGB\').copy()\n    conditions = load_conditions(directory)\n    records = None\n    if resume:\n        from .resume import load_records, complete_arm, archive_partial, rgb_hash\n        if (out/\'input.png\').exists() and rgb_hash(out/\'input.png\') != rgb_hash(directory/\'input.png\'):\n            raise ValueError(\'Cached audit input differs from requested source\')\n        records = load_records(out, conditions)\n    if records is None:\n        records = fit_records(source, conditions, extractor, out / \'extraction_input\')\n    source_rgb = image_tensor(source, next(extractor.parameters()).device)\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    with torch.no_grad():\n        controller.refresh_erase(source_rgb)  # Diagnostic only; first restore step clears erasure.\n    support = fixed_support(source, controller)\n    np.savez_compressed(out / \'fixed_diagnostic_regions.npz\', **support)\n    # Reset the controller: diagnostic setup must not change the repair experiment.\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    reference = reference_sheet(conditions, source.width)\n    reference.save(out / \'reference_diagnostic_only.png\')\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor, return_dict=False)[0].float()/2+.5\n    def encode(rgb):\n        return pipe.vae.encode(rgb.float()*2-1).latent_dist.mode()*pipe.vae.config.scaling_factor\n    def save_tensor(rgb, path):\n        a = rgb[0].detach().permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(a*255).astype(np.uint8)).save(path)\n    # Same frozen float32 VAE and posterior-mode encoding for every compared arm.\n    # This is deliberately recorded: legacy cyclic used prepare_latents\' sampled posterior.\n    with torch.no_grad():\n        clean_input = encode(source_rgb)\n        save_tensor(decode(clean_input), out / \'vae_roundtrip.png\')\n        positive, negative = pipe.encode_prompt(case[\'prompt\'], \'cuda\', 1, True,\n                                               negative_prompt=case[\'negative_prompt\'])\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        noise = torch.randn(clean_input.shape, generator=torch.Generator(\'cpu\').manual_seed(seed),\n                            dtype=torch.float32).to(clean_input.device)\n        initial = pipe.scheduler.add_noise(clean_input, noise, timesteps[:1])\n    initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n    legacy_initial_unrecorded = False\n    if resume:\n        signature = dict(steps=steps, strength=strength, seed=seed, cfg_scale=cfg_scale,\n            boundary_config=boundary_config, width_config=width_config,\n            input_rgb_sha256=rgb_hash(directory/\'input.png\'))\n        cache = out/\'resume_state.json\'\n        if cache.exists():\n            previous_state = json.loads(cache.read_text(encoding=\'utf-8\'))\n            if previous_state[\'signature\'] != signature:\n                raise ValueError(\'Resume settings/input differ from saved state\')\n            with np.load(out/\'resume_initial_latent.npz\', allow_pickle=False) as a:\n                initial = torch.as_tensor(a[\'initial\'].copy(), device=clean_input.device)\n            initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n            if initial_hash != previous_state[\'initial_latent_sha256\']:\n                raise ValueError(\'Cached initial latent is damaged\')\n            legacy_initial_unrecorded = previous_state[\'legacy_initial_unrecorded\']\n        else:\n            old_report = out/\'report.json\'\n            if old_report.exists():\n                expected = json.loads(old_report.read_text(encoding=\'utf-8\'))[\'configuration\'][\'initial_latent_sha256\']\n                if expected != initial_hash:\n                    raise ValueError(\'Historical initial latent differs. Completed cases can be reused, but partial cases need matching runtime or a fresh comparison.\')\n            else:\n                legacy_initial_unrecorded = any(complete_arm(out, a, [int(t) for t in timesteps])\n                                                for a in (\'redenoise_only\',\'cyclic_guided\',\'boundary_guarded\',\'observed_width\'))\n            np.savez_compressed(out/\'resume_initial_latent.npz\', initial=initial.cpu().numpy())\n            json_write(cache, dict(signature=signature, initial_latent_sha256=initial_hash,\n                legacy_initial_unrecorded=legacy_initial_unrecorded))\n    # Expose reconstruction before the actual rerun so failures still retain evidence.\n    source.save(out / \'input.png\')\n    traces = {}\n    arms = [(\'redenoise_only\', None), (\'cyclic_guided\', controller)]\n    if boundary_guard:\n        from stroke_model.boundary_guarded_repair import BoundaryGuardConfig, BoundaryGuardedRepair\n        guard_config = BoundaryGuardConfig(**(boundary_config or {}))\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        with torch.no_grad():\n            guarded.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'boundary_initial_maps.npz\',\n            erase=guarded.erase.cpu().numpy(), restore=guarded.restore.cpu().numpy(),\n            preserve=guarded.preserve.cpu().numpy())\n        json_write(out/\'boundary_initial_diagnostics.json\', guarded.boundary_history)\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        arms.append((\'boundary_guarded\', guarded))\n    if observed_width:\n        from stroke_model.observed_width_repair import ObservedWidthConfig, ObservedWidthRepair\n        def calibrated_controller():\n            return ObservedWidthRepair(source_rgb, extractor, records,\n                boundary_config=guard_config, width_config=ObservedWidthConfig(**(width_config or {})))\n        calibrated = calibrated_controller()\n        with torch.no_grad():\n            calibrated.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'observed_width_initial_maps.npz\',\n            erase=calibrated.erase.cpu().numpy(), restore=calibrated.restore.cpu().numpy(),\n            preserve=calibrated.preserve.cpu().numpy())\n        json_write(out/\'observed_width_calibration.json\', calibrated.metadata)\n        json_write(out/\'observed_width_initial_diagnostics.json\', calibrated.boundary_history)\n        arms.append((\'observed_width\', calibrated_controller()))\n    if width_probe:\n        alternate = width_probe_conditions(conditions)\n        alt_records = fit_records(source, alternate, extractor, out / \'width_probe_input\')\n        arms.append((\'reference_width_probe\', RestoreSettleErase(source_rgb, extractor, alt_records)))\n    for arm, objective in arms:\n        if resume and complete_arm(out, arm, [int(t) for t in timesteps]):\n            traces[arm] = json.loads((out/(arm+\'_trace.json\')).read_text(encoding=\'utf-8\'))\n            if progress:\n                progress(case[\'case_id\'], arm, len(timesteps), len(timesteps),\n                         dict(phase=\'cached\', accepted_inner_steps=0))\n            continue\n        if resume:\n            archive_partial(out, arm)\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.clone()\n        trace = []\n        frames = out / arm\n        frames.mkdir()\n        for index, t in enumerate(timesteps):\n            with torch.no_grad():\n                model_input = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(model_input.to(dtype=next(pipe.unet.parameters()).dtype), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                base = scheduler.step((u+cfg_scale*(c-u)).float(), t, latent, eta=0., return_dict=True)\n            clean, latent = base.pred_original_sample.detach(), base.prev_sample.detach()\n            row = dict(step=index, timestep=int(t), phase=\'baseline\', accepted_inner_steps=0)\n            if objective is not None:\n                with torch.no_grad():\n                    row.update(objective.begin_step(decode(clean), index))\n                if row[\'phase\'] == \'restore\':\n                    clean, projection = objective.project_clean(clean, decode, encode)\n                    row.update(projection)\n                clean, optimization = optimize_clean(clean, decode, objective,\n                    iterations=2 if row[\'phase\'] in (\'restore\', \'settle\') else 4,\n                    step_ratio=.008, budget_ratio=.06, backtracks=5)\n                row.update(optimization)\n                with torch.no_grad():\n                    objective.finish_step(decode(clean), row)\n                previous = int(t) - scheduler.config.num_train_timesteps // scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[previous] if previous >= 0 else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent + alpha.sqrt() * (clean-base.pred_original_sample)\n                for name in (\'erase\', \'restore\', \'preserve\'):\n                    value = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                    Image.fromarray(np.rint(value.clip(0, 1)*255).astype(np.uint8)).save(frames / f\'{name}_{index:03d}.png\')\n            with torch.no_grad():\n                save_tensor(decode(clean), frames / f\'clean_prediction_{index:03d}.png\')\n            latent = latent.detach()\n            trace.append(row)\n            if progress:\n                progress(case[\'case_id\'], arm, index+1, len(timesteps), row)\n        with torch.no_grad():\n            save_tensor(decode(latent), out / (arm+\'.png\'))\n        traces[arm] = trace\n        json_write(out / (arm+\'_trace.json\'), trace)\n        if objective is not None and hasattr(objective, \'boundary_history\'):\n            json_write(out/(arm+\'_map_history.json\'), objective.boundary_history)\n        if on_arm_complete:\n            on_arm_complete(case[\'case_id\'], arm)\n    results = {\'input\': source, \'vae_roundtrip\': Image.open(out / \'vae_roundtrip.png\').convert(\'RGB\')}\n    for arm, _ in arms:\n        results[arm] = Image.open(out / (arm+\'.png\')).convert(\'RGB\')\n    metric_rows = []\n    for arm, result in results.items():\n        # Save final extraction for VISUAL review; masks are not quality probabilities.\n        if arm != \'input\':\n            fit_records(result, conditions, extractor, out / (\'extraction_\'+arm), fit=False)\n        for slot in range(4):\n            half = source.width // 2\n            x, y = slot % 2 * half, slot // 2 * half\n            box = (x, y, x+half, y+half)\n            local = {k: v[y:y+half, x:x+half] for k, v in support.items()}\n            for threshold in (.35, .5, .65):\n                values = measure(source.crop(box), result.crop(box), reference.crop(box), local, threshold)\n                if boundary_guard:\n                    # Include all source foreground, including tips excluded by\n                    # the original automatic healthy mask. This still isn\'t GT.\n                    ink = ink_array(source.crop(box))\n                    all_source = dict(healthy=ink > .5, center=skeletonize(ink > .5))\n                    full = measure(source.crop(box), result.crop(box), reference.crop(box), all_source, threshold)\n                    values.update({\'all_source_\'+k:full[k] for k in\n                        (\'fixed_center_width_ratio_proxy\',\'healthy_ink_drop\',\'healthy_ink_retention\',\'healthy_ink_mse\')})\n                metric_rows.append(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot+1, arm=arm, **values))\n    json_write(out / \'report.json\', dict(case=case, metrics=metric_rows, traces=traces,\n        configuration=dict(steps=steps, actual_steps=len(timesteps), timesteps=[int(t) for t in timesteps],\n            strength=strength, seed=seed, cfg_scale=cfg_scale, vae_dtype=\'float32\',\n            vae_posterior=\'mode_shared_by_all_arms\' if not legacy_initial_unrecorded else \'legacy_initial_not_recorded\',\n            paired_initial_latent=not legacy_initial_unrecorded,\n            resume=resume, legacy_initial_unrecorded=legacy_initial_unrecorded,\n            initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n            width_probe=width_probe, original_repair_parameters=True,\n            boundary_guard=boundary_guard, boundary_config=boundary_config if boundary_guard else None,\n            observed_width=observed_width, width_config=width_config if observed_width else None,\n            legacy_difference=\'Posterior mode, not legacy stochastic VAE encoding\'),\n        limitations=[\'Widths are fixed-source-center distance proxies; movement is a confound.\',\n            \'Healthy masks are heuristic and require human review.\',\n            \'Reference similarity is not correctness and cannot prove template copying.\',\n            \'No automatic pass/fail or independent text accuracy claim.\',\n            \'Width probe is synthetic dilation and may be outside extractor training distribution.\']))\n    make_figures(out, source, reference, results, support)\n    return metric_rows\n\n\ndef make_figures(out, source, reference, results, support):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    labels = list(results)\n    fig, axes = plt.subplots(1, len(labels)+1, figsize=(4*(len(labels)+1), 4), squeeze=False)\n    for ax, (name, image) in zip(axes.flat, [(\'reference (diagnostic)\', reference), *results.items()]):\n        ax.imshow(image); ax.set_title(name); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'comparison.png\', dpi=120); plt.close(fig)\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4))\n    for ax, key in zip(axes, (\'healthy\', \'center\', \'initial_edit\')):\n        ax.imshow(source); ax.imshow(np.ma.masked_where(~support[key], support[key]), cmap=\'autumn\', alpha=.6)\n        ax.set_title(key+\' (heuristic)\'); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'diagnostic_regions.png\', dpi=120); plt.close(fig)\n    for slot in range(1, 5):\n        with np.load(out / \'extraction_input\' / f\'slot_{slot}.npz\') as a:\n            count = len(a[\'extracted_masks\'])\n        fig, axes = plt.subplots(len(results), count+2,\n            figsize=(2.2*(count+2), 2.3*len(results)), squeeze=False)\n        half=source.width//2; x=(slot-1)%2*half; y=(slot-1)//2*half\n        for row, (arm, image) in enumerate(results.items()):\n            with np.load(out / (\'extraction_\'+arm) / f\'slot_{slot}.npz\') as a:\n                images = [(\'reference\', 1-a[\'reference\'][0]),\n                          (arm, np.asarray(image.crop((x,y,x+half,y+half))))] + [\n                    (f\'stroke {i+1}\', 1-m) for i,m in enumerate(a[\'extracted_masks\'])]\n            for ax, (name, value) in zip(axes[row], images):\n                ax.imshow(value, cmap=\'gray\', vmin=0, vmax=1) if value.ndim==2 else ax.imshow(value)\n                ax.set_title(name); ax.axis(\'off\')\n        fig.tight_layout(); fig.savefig(out / f\'extraction_slot_{slot}.png\', dpi=110); plt.close(fig)\n\n\ndef write_tables(root, cases, metrics, failures):\n    root = Path(root)\n    if metrics:\n        with (root / \'automatic_metrics.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(metrics[0]))\n            writer.writeheader(); writer.writerows(metrics)\n    fields = [\'case_id\', \'kind\', \'slot\', \'arm\', \'target_character\', \'input_error_type\',\n              \'extraction_ok\', \'error_reduced\', \'thinner\', \'new_error\', \'style_changed\', \'notes\']\n    with (root / \'human_review.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for case in cases:\n            arms = sorted({r[\'arm\'] for r in metrics if r[\'case_id\'] == case[\'case_id\']})\n            for slot in range(1, 5):\n                for arm in arms:\n                    writer.writerow(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot, arm=arm))\n    json_write(root / \'failures.json\', failures)\n', 'repair_audit/target_guided_repair.py': '"""Connect the existing 2x2 generator/extractor to an aligned target-image guide.\n\nThe CLI prepares and inspects targets on CPU. The Python repair_generated_sheet\nAPI uses an already loaded pipeline/extractor for actual img2img re-denoising.\n"""\nimport argparse\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport torch\nfrom torch.nn import functional as F\n\nfrom stroke_model.target_stroke_plan import build_target_plan\nfrom .stage1 import read_archive, load_conditions, json_write\n\n\ndef extract_sheet_masks(source,conditions,extractor):\n    device=next(extractor.parameters(),torch.empty(0)).device\n    rgb=torch.as_tensor(np.asarray(source.convert(\'RGB\')).copy(),device=device).permute(2,0,1)[None].float()/255\n    ink=1-(rgb*rgb.new_tensor([.299,.587,.114])[None,:,None,None]).sum(1,keepdim=True)\n    side=source.width//2;result=[]\n    extractor.eval().requires_grad_(False)\n    with torch.no_grad():\n        for gi,condition in enumerate(conditions):\n            x0=(gi%2)*side;y0=(gi//2)*side\n            ref=torch.as_tensor(condition[\'reference\'],device=device)[None].float()\n            strokes=torch.as_tensor(condition[\'strokes\'],device=device)[None].float()\n            crop=F.interpolate(ink[...,y0:y0+side,x0:x0+side],ref.shape[-2:],mode=\'bilinear\',align_corners=False)\n            result.append(extractor(crop,ref,strokes)[\'masks\'][0].float().cpu().numpy())\n    return result\n\n\ndef save_plan(guide,rendered,output):\n    out=Path(output);out.mkdir(parents=True,exist_ok=False)\n    def image(array):return Image.fromarray(np.rint(array*255).clip(0,255).astype(\'uint8\'))\n    image(guide.source).save(out/\'input.png\');image(guide.target).save(out/\'aligned_target.png\')\n    image(guide.weights).save(out/\'edit_strength.png\')\n    preview=guide.project(torch.tensor(guide.source.copy()).permute(2,0,1)[None])\n    image(preview[0].permute(1,2,0).numpy()).save(out/\'cpu_blend_diagnostic_only.png\')\n    for gi,item in enumerate(rendered,1):np.savez_compressed(out/f\'glyph_{gi}_target_strokes.npz\',**item)\n    arrays=dict(target=guide.target,weights=guide.weights,source=guide.source)\n    if getattr(guide,\'background\',None) is not None:arrays[\'background\']=guide.background\n    np.savez_compressed(out/\'guide.npz\',**arrays)\n    json_write(out/\'plan.json\',dict(**guide.report,guidance=guide.metadata,diffusion_run=False))\n    ink_only = guide.metadata[\'kind\']==\'glyph_ink_projection\'\n    names=[(\'generated input\',\'input.png\'),(\'width-rendered + aligned target\',\'aligned_target.png\'),\n           (\'glyph edit mask\' if ink_only else \'score-controlled edit strength\',\'edit_strength.png\'),\n           (\'ink projection preview\' if ink_only else \'CPU blend: NOT diffusion\',\'cpu_blend_diagnostic_only.png\')]\n    cell=288;canvas=Image.new(\'RGB\',(cell*4,cell+35),\'white\');draw=ImageDraw.Draw(canvas)\n    for i,(title,name) in enumerate(names):\n        draw.text((i*cell+5,9),title,fill=\'black\')\n        with Image.open(out/name) as im:canvas.paste(im.convert(\'RGB\').resize((cell,cell)),(i*cell,35))\n    canvas.save(out/\'comparison.png\')\n\n\ndef repair_generated_sheet(pipe,extractor,source,conditions,output,*,cached_masks=None,scores=None,\n                           config=None,registration_config=None,steps=60,strength=.25,seed=2026,\n                           prompt=\'\',negative_prompt=\'\',progress=None,save_step_images=False,ink_only=False,\n                           background_image=None):\n    """Call directly after existing generate_cases / generation of a 2x2 sheet."""\n    from diffusion_model.stroke_projected_diffusion import run_projected_pair\n    if background_image is not None and not ink_only:\n        raise ValueError(\'Background compositing requires ink-only guidance\')\n    masks=cached_masks if cached_masks is not None else extract_sheet_masks(source,conditions,extractor)\n    if ink_only and registration_config is None:\n        from stroke_model.source_structure import SourceConfig\n        registration_config=SourceConfig(rotation=0.,shear=0.)\n    guide,rendered=build_target_plan(source,conditions,masks,scores=scores,config=config,\n                                    registration_config=registration_config)\n    if ink_only:\n        from stroke_model.ink_target_guide import InkTargetGuide\n        guide=InkTargetGuide(guide,rendered,background_image)\n        if background_image is not None:\n            source=Image.fromarray(np.rint(guide.source*255).astype(\'uint8\'))\n    root=Path(output);root.mkdir(parents=True,exist_ok=False)\n    save_plan(guide,rendered,root/\'target_plan\')\n    report=run_projected_pair(pipe,source,guide,root/\'reverse\',steps=steps,strength=strength,seed=seed,\n        prompt=prompt,negative_prompt=negative_prompt,start_from_noise=False,save_final_projection=False,\n        progress=progress,save_step_images=save_step_images)\n    json_write(root/\'pipeline.json\',dict(stages=[\'existing_2x2_generation\',\'stroke_extraction\',\n        \'observed_width_rendering\',\'shared_glyph_affine\',\'ink_projection\' if ink_only else \'score_to_edit_plan\',\n        \'source_initialized_rediffusion\'],\n        score_source=\'none_for_ink_projection\' if ink_only else (\'provided\' if scores is not None else \'uncalibrated_geometry_proxy_v1\'),\n        new_training=False,source_content_used=report[\'source_content_used\'],final_pixel_replacement=False,\n        plan=\'target_plan/plan.json\',diffusion=\'reverse/report.json\'))\n    return guide,report\n\n\ndef _cached_masks(z,case,source,conditions):\n    prefix=case[\'case_id\'];result=[]\n    for slot,condition in enumerate(conditions,1):\n        name=f\'{prefix}/audit/extraction_input/slot_{slot}.npz\'\n        if name not in z.namelist():return None\n        with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:\n            for key in [\'reference\',\'strokes\']:\n                if key in a and not np.array_equal(a[key],condition[key]):raise ValueError(\'Cached conditions differ\')\n            result.append(a[\'extracted_masks\'].copy())\n    with Image.open(io.BytesIO(z.read(prefix+\'/input.png\'))) as original:\n        if not np.array_equal(np.asarray(original.convert(\'RGB\')),np.asarray(source)):\n            raise ValueError(\'Cached extraction uses different source pixels\')\n    return result\n\n\ndef prepare_archive(archive,output,*,case_ids=None,observations_archive=None,scores=None,config=None):\n    root=Path(output)\n    cases,_=read_archive(archive,root,case_ids,include_controls=True)\n    results=[]\n    with zipfile.ZipFile(archive) as z:\n        for case in cases:\n            directory=Path(case[\'directory\'])\n            with Image.open(directory/\'input.png\') as image:source=image.convert(\'RGB\').copy()\n            conditions=load_conditions(directory);masks=_cached_masks(z,case,source,conditions)\n            if masks is None and observations_archive:\n                with zipfile.ZipFile(observations_archive) as previous:masks=_cached_masks(previous,case,source,conditions)\n            if masks is None:\n                results.append(dict(case_id=case[\'case_id\'],status=\'deferred_no_saved_extraction\',diffusion_run=False))\n                continue\n            try:\n                given=None if scores is None else scores.get(case[\'case_id\'])\n                guide,rendered=build_target_plan(source,conditions,masks,scores=given,config=config)\n                save_plan(guide,rendered,directory/\'target_plan\')\n                results.append(dict(case_id=case[\'case_id\'],status=\'target_ready_cpu_only\',\n                    strokes=len(guide.report[\'strokes\']),selected_strokes=sum(s[\'edit_strength\']>0 for s in guide.report[\'strokes\']),\n                    score_source=\'provided\' if given is not None else \'geometry_proxy_v1\',diffusion_run=False))\n            except ValueError as exc:\n                results.append(dict(case_id=case[\'case_id\'],status=\'deferred_invalid_plan\',error=str(exc),diffusion_run=False))\n            json_write(root/\'target_plan_summary.json\',results)\n    json_write(root/\'target_plan_summary.json\',results)\n    return results\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--archive\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--observations-archive\');parser.add_argument(\'--case\',action=\'append\',dest=\'cases\')\n    parser.add_argument(\'--scores\',help=\'JSON: case_id -> four lists of consistency scores in [0,100], in stroke-ID order\')\n    args=parser.parse_args()\n    given=json.loads(Path(args.scores).read_text(encoding=\'utf-8\')) if args.scores else None\n    print(json.dumps(prepare_archive(args.archive,args.output,case_ids=args.cases,\n        observations_archive=args.observations_archive,scores=given),indent=2))\n', 'repair_audit/fresh_target_inputs.py': '"""Create repeatable whole-sheet img2img inputs from saved canonical strokes."""\nimport math\nfrom pathlib import Path\nimport shutil\n\nimport numpy as np\nfrom PIL import Image\nimport torch\n\nfrom .stage1 import load_conditions, json_write\n\n\ndef canonical_sheet(conditions, size):\n    if len(conditions) != 4 or size % 16:\n        raise ValueError(\'Expected four conditions and a sheet size divisible by 16\')\n    sheet = Image.new(\'RGB\', (size, size), \'white\')\n    for i, condition in enumerate(conditions):\n        reference = np.asarray(condition[\'reference\'])[0]\n        glyph = Image.fromarray(np.rint((1-reference)*255).clip(0,255).astype(\'uint8\'))\n        sheet.paste(glyph.convert(\'RGB\').resize((size//2,size//2),Image.Resampling.BICUBIC),\n                    ((i%2)*(size//2),(i//2)*(size//2)))\n    return sheet\n\n\ndef generate_fresh_cases(pipe, seed_cases, output, *, strengths=(.65,), seeds=(100,101),\n                         steps=60, cfg_scale=7.5, on_saved=None):\n    """One diffusion call sees all four glyphs. Keep every output, even failures.\n\n    Saved ZIPs supply reference conditions, not a pool of pre-repaired images.\n    The original canonical sheet is also returned as a normal preservation control.\n    """\n    strengths, seeds = list(strengths), list(seeds)\n    if (not strengths or not seeds or type(steps) is not int or steps<2\n        or any(not math.isfinite(s) or not 0<s<1 or int(steps*s)<1 for s in strengths)\n        or any(type(s) is not int or s<0 for s in seeds)\n        or len(set(strengths))!=len(strengths) or len(set(seeds))!=len(seeds)):\n        raise ValueError(\'Invalid or duplicate generation settings\')\n    controls = [c for c in seed_cases if c[\'kind\']==\'normal_control\']\n    if not controls: raise ValueError(\'No canonical control groups in the source ZIP\')\n    root = Path(output); root.mkdir(parents=True,exist_ok=False)\n    cases=[]\n    prompt = (\'a typography sheet with 4 Korean Hangul glyphs arranged in 2 rows and 2 columns, \'\n              \'black ink on plain white background, flat front view, no grid lines, \'\n              \'preserve the glyphs and layout of the input image\')\n    negative = \'photograph, scenery, paper texture, shadows, colored background, grid lines, decorations\'\n    report = dict(kind=\'canonical_joint_sheet_img2img\',strengths=strengths,seeds=seeds,\n        steps=steps,cfg_scale=cfg_scale,prompt=prompt,negative_prompt=negative,\n        filtering=\'none; wrong, displaced and empty glyphs are retained\',cases=cases)\n    def save(image,reference_case,case_id,**generation):\n        directory=root/case_id;directory.mkdir()\n        image.save(directory/\'input.png\')\n        for slot in range(1,5):\n            shutil.copy2(Path(reference_case[\'directory\'])/f\'condition_{slot}.npz\',directory/f\'condition_{slot}.npz\')\n        item=dict(reference_case,case_id=case_id,directory=str(directory),\n                  kind=\'normal_control\' if generation[\'control\'] else \'generated\',\n                  prompt=prompt,negative_prompt=negative,generation=generation,\n                  source_reference_case=reference_case[\'case_id\'],reused_stage1_original=False)\n        json_write(directory/\'input_metadata.json\',item)\n        cases.append(item);json_write(root/\'generation_report.json\',report)\n        if on_saved: on_saved(item,cases)\n    for control in controls:\n        with Image.open(Path(control[\'directory\'])/\'input.png\') as old: size=old.width\n        # A source font image may differ from the ONE fixed repair prototype.\n        generation_image=Path(control[\'directory\'])/\'generation_reference_sheet.png\'\n        if generation_image.is_file():\n            with Image.open(generation_image) as image: initial=image.convert(\'RGB\').copy()\n            if initial.size!=(size,size):raise ValueError(\'Invalid generation reference dimensions\')\n        else: initial=canonical_sheet(load_conditions(control[\'directory\']),size)\n        group=control[\'case_id\'].removesuffix(\'_normal_control\')\n        save(initial,control,group+\'_normal_control\',control=True,strength=None,seed=None)\n        for si,strength in enumerate(strengths):\n            for ri,seed in enumerate(seeds):\n                case_id=f\'{group}_page_{100+si*len(seeds)+ri:03d}\'\n                print(\'Joint 2x2 generation:\',case_id,\'strength=\',strength,\'seed=\',seed,flush=True)\n                generated=pipe(prompt=prompt,negative_prompt=negative,image=initial,\n                    strength=float(strength),num_inference_steps=steps,guidance_scale=cfg_scale,\n                    generator=torch.Generator(\'cpu\').manual_seed(seed))\n                image=generated.images[0]\n                if image.size!=initial.size:raise ValueError(\'Generation changed fixed sheet dimensions\')\n                flags=getattr(generated,\'nsfw_content_detected\',None)\n                save(image,control,case_id,control=False,strength=strength,seed=seed,\n                     safety_filtered=bool(flags[0]) if flags is not None else None)\n    return cases\n', 'diffusion_model/pilot_data.py': '"""Recover the transient stroke-training data for the diffusion scoring pilot."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom stroke_model.convert_reference_csv import existing_path\nfrom stroke_model.prepare_stroke_zips import prepare_stroke_zips\n\n\ndef _read_json(path):\n    return json.loads(path.read_text(encoding=\'utf-8\')) if path.is_file() else {}\n\n\ndef _usable_manifest(path):\n    if not path.is_file():\n        return False\n    rows = _read_json(path)\n    return (isinstance(rows, list) and bool(rows) and any(r.get(\'split\') == \'train\' for r in rows)\n            and all(((path.parent / row[\'path\']).is_file() if row.get(\'path\') else\n                     Path(row[\'reference\']).is_file() and all(Path(p).is_file() for p in row[\'reference_strokes\']))\n                    for row in rows))\n\n\ndef prepare_pilot_data(checkpoint, data_root=None, *, csv_path=None,\n                       local_root=\'/content/hsqm_stroke_zip\', manifest=None, reference_only=True):\n    """Default to reference-only preparation; retain full conversion as opt-in.\n\n    Reads the CSV snapshot, seed, resolution and supervision saved beside best.pt.\n    Does not train a model or modify checkpoint files. No whole-Drive search.\n    """\n    checkpoint = existing_path(checkpoint)\n    run = checkpoint.parent\n    config = _read_json(run/\'run_config.json\')\n    recorded = config.get(\'manifest\')\n    candidates = [Path(manifest)] if manifest is not None else ([Path(recorded)] if recorded else [])\n    for candidate in candidates:\n        if reference_only and manifest is None and candidate.is_file():\n            recorded_rows=_read_json(candidate)\n            if recorded_rows and recorded_rows[0].get(\'format\')!=\'reference_paths_v1\':\n                continue\n        if _usable_manifest(candidate):\n            print(\'현재 세션의 학습 데이터 재사용:\', candidate, flush=True)\n            return dict(manifest=str(candidate.resolve()), mode=\'reuse_live_training_data\',\n                        checkpoint=str(checkpoint.resolve()))\n    if manifest is not None:\n        print(\'지정한 학습 목록 또는 연결된 NPZ가 없어 원본 압축 자료로 다시 준비합니다.\', flush=True)\n    conversion = _read_json(run/\'conversion_report.json\')\n    if \'image_size\' not in config or \'seed\' not in config:\n        raise ValueError(\'체크포인트 옆 run_config.json에 image_size/seed가 필요합니다. \'\n                         \'기존 학습 결과 폴더의 best.pt를 선택하세요.\')\n    supervision = conversion.get(\'supervision\')\n    if supervision not in (\'actual_input_multihot\', \'exclusive_annotated\'):\n        raise ValueError(\'체크포인트 옆 conversion_report.json에서 기존 데이터 변환 방식을 확인할 수 없습니다.\')\n    # A saved CSV is the precise input to the original preparation, unlike a new\n    # CSV guessed from another dataset or a numeric character label.\n    snapshot = run/\'selected_paths.csv\'\n    if csv_path is not None:\n        selected_csv = existing_path(csv_path)\n        if snapshot.is_file() and selected_csv.read_bytes() != snapshot.read_bytes():\n            raise ValueError(\'CSV_PATH가 체크포인트의 학습 CSV와 다릅니다. 같은 CSV를 사용하세요.\')\n    elif snapshot.is_file():\n        selected_csv = snapshot\n    else:\n        raise FileNotFoundError(\'기존 학습 결과 폴더에 selected_paths.csv가 없습니다. \'\n                                \'CSV_PATH에 해당 학습에 쓴 원본 CSV를 지정하세요.\')\n\n    # Saved preparation metadata contains original archive paths. Infer their\n    # common dataset root only when the caller did not specify one.\n    if data_root is None:\n        import os\n        complete = _read_json(run/\'complete.json\')\n        archive_paths = [str(item[0]) for item in complete.get(\'archives\', [])]\n        if not archive_paths:\n            raise ValueError(\'DATA_ROOT에 기존 학습 ZIP/TAR가 있는 폴더를 지정하세요.\')\n        data_root = Path(os.path.commonpath(archive_paths))\n        if len(archive_paths) == 1:\n            data_root = data_root.parent\n    source = existing_path(data_root)\n    if reference_only:\n        from stroke_model.reference_only_data import prepare_reference_catalog\n        catalog=prepare_reference_catalog(source,selected_csv,conversion,int(config[\'image_size\']),\n                                          int(config[\'seed\']),local_root)\n        return dict(manifest=str(catalog.resolve()),mode=\'reference_only_no_training_npz\',\n            checkpoint=str(checkpoint.resolve()),data_root=str(source.resolve()),\n            csv_snapshot=str(selected_csv.resolve()),\n            csv_sha256=hashlib.sha256(selected_csv.read_bytes()).hexdigest(),\n            image_size=int(config[\'image_size\']),seed=int(config[\'seed\']),\n            supervision=supervision,original_image_content_fingerprint_available=False)\n    print(\'기존 임시 manifest 없음 → 학습 ZIP/TAR·CSV로 자동 재생성\', flush=True)\n    print(\'학습 압축 자료:\', source, flush=True)\n    print(\'원래 학습 CSV:\', selected_csv, flush=True)\n    print(\'기존 설정: size=\', config[\'image_size\'], \'seed=\', config[\'seed\'],\n          \'supervision=\', supervision, flush=True)\n    rebuilt = prepare_stroke_zips(source, local_root=local_root, csv_path=selected_csv,\n                                  size=int(config[\'image_size\']), seed=int(config[\'seed\']),\n                                  independent=supervision == \'actual_input_multihot\')\n    rebuilt_conversion = _read_json(rebuilt.parent/\'conversion_report.json\')\n    # Catch changed/missing source samples before claiming the restored split is\n    # the original training population. Image identity itself was not fingerprinted\n    # by old training runs and cannot be retroactively guaranteed.\n    for key in (\'supervision\', \'group_column\', \'skipped_count\'):\n        if key in conversion and conversion[key] != rebuilt_conversion.get(key):\n            raise ValueError(f\'재생성 데이터의 {key}가 원래 학습 기록과 다릅니다.\')\n    if \'rows\' in conversion and len(conversion[\'rows\']) != len(rebuilt_conversion.get(\'rows\', [])):\n        raise ValueError(\'재생성한 표본 수가 원래 학습 기록과 다릅니다.\')\n    if not _usable_manifest(rebuilt):\n        raise ValueError(\'재생성한 학습 목록/NPZ를 확인할 수 없습니다.\')\n    result = dict(manifest=str(rebuilt.resolve()), mode=\'rebuilt_from_training_archives\',\n                  checkpoint=str(checkpoint.resolve()), data_root=str(source.resolve()),\n                  csv_snapshot=str(selected_csv.resolve()),\n                  csv_sha256=hashlib.sha256(selected_csv.read_bytes()).hexdigest(),\n                  image_size=int(config[\'image_size\']), seed=int(config[\'seed\']),\n                  supervision=supervision, original_image_content_fingerprint_available=False)\n    print(\'학습 목록 준비 완료:\', rebuilt, flush=True)\n    return result\n', 'diffusion_model/score_pilot.py': '"""Pretrained img2img outputs on training references, then unchanged source scores.\n\nThis is a domain-transfer pilot, not text-to-image accuracy or guidance training.\n"""\nfrom datetime import datetime\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef load_catalog_reference(entry):\n    """Read one selected glyph, from either a training NPZ or reference PNGs."""\n    if entry.get(\'path\'):\n        with np.load(entry[\'path\'],allow_pickle=False) as data:\n            reference,strokes=[data[key].astype(np.float32) for key in (\'reference\',\'strokes\')]\n    else:\n        from stroke_model.convert_reference_csv import load_ink\n        reference=load_ink(entry[\'reference\'],entry[\'image_size\'])\n        strokes=np.concatenate([load_ink(p,entry[\'image_size\']) for p in entry[\'reference_strokes\']])\n    if (reference.ndim!=3 or reference.shape[0]!=1 or reference.shape[1]!=reference.shape[2]\n            or strokes.ndim!=3 or strokes.shape[1:]!=reference.shape[1:] or not len(strokes)):\n        raise ValueError(\'Invalid reference/stroke shapes\')\n    for array in (reference,strokes):\n        if not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n            raise ValueError(\'Expected finite absolute ink in [0,1]\')\n    if reference.max()<=.5 or (strokes.max(axis=(1,2))<=.5).any():\n        raise ValueError(\'Empty reference or stroke\')\n    return reference,strokes\n\n\ndef training_catalog(manifest):\n    """Only explicit train rows; never infer characters from numeric image IDs."""\n    manifest = Path(manifest).resolve()\n    rows = json.loads(manifest.read_text(encoding=\'utf-8\'))\n    if not isinstance(rows, list):\n        raise ValueError(\'Expected the stroke-training manifest JSON list\')\n    groups = {}\n    catalog, seen = [], set()\n    for index, row in enumerate(rows):\n        split = row.get(\'split\')\n        if split not in (\'train\', \'val\', \'test\'):\n            raise ValueError(\'Every manifest row needs train/val/test split\')\n        if row.get(\'group_id\'):\n            previous = groups.setdefault(row[\'group_id\'], split)\n            if previous != split:\n                raise ValueError(\'Training manifest contains group leakage\')\n        if split != \'train\':\n            continue\n        if row.get(\'format\') == \'reference_paths_v1\':\n            digest=row[\'reference_sha256\']\n            if digest in seen:continue\n            seen.add(digest)\n            catalog.append({**row,\'manifest_row\':index+1})\n            continue\n        path = (manifest.parent / row[\'path\']).resolve()\n        with np.load(path, allow_pickle=False) as data:\n            reference, strokes = [data[key].astype(np.float32) for key in (\'reference\', \'strokes\')]\n        if (reference.ndim != 3 or reference.shape[0] != 1 or reference.shape[1] != reference.shape[2]\n                or strokes.ndim != 3 or strokes.shape[1:] != reference.shape[1:] or not len(strokes)):\n            raise ValueError(f\'Invalid reference/strokes shapes: {path}\')\n        for array in (reference, strokes):\n            if not np.isfinite(array).all() or array.min() < 0 or array.max() > 1:\n                raise ValueError(f\'Expected finite absolute ink in [0,1]: {path}\')\n        if reference.max() <= .5 or (strokes.max(axis=(1, 2)) <= .5).any():\n            raise ValueError(f\'Empty reference or stroke: {path}\')\n        digest = hashlib.sha256(reference.tobytes() + strokes.tobytes()).hexdigest()\n        if digest in seen:\n            continue\n        seen.add(digest)\n        catalog.append(dict(manifest_row=index + 1, path=str(path),\n                            char_id=str(row.get(\'char_id\', \'\')), reference_sha256=digest,\n                            source_image=str(row.get(\'source_image\',\'\'))))\n    if not catalog:\n        raise ValueError(\'No train references in this manifest\')\n    return catalog\n\n\ndef scored_ink(image, size):\n    """Fixed black-on-white luminance conversion; no polarity guesses or cropping."""\n    rgba = image.convert(\'RGBA\')\n    white = Image.new(\'RGBA\', rgba.size, (255, 255, 255, 255))\n    gray = Image.alpha_composite(white, rgba).convert(\'L\')\n    gray = gray.resize((size, size), Image.Resampling.BILINEAR)\n    return (1 - np.asarray(gray, dtype=np.float32) / 255)[None].copy()\n\n\ndef generate_pilot(manifest, manifest_rows, output, *, strengths=(.25, .45, .65),\n                   seeds=(100, 101), steps=30,\n                   model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n                   pipeline=None):\n    """Save every output, including wrong letters, noise and safety-filter flags.\n\n    pipeline injection is for orchestration tests, never evidence of SD quality.\n    The caller must supply the manifest used for the selected stroke checkpoint.\n    """\n    catalog = {entry[\'manifest_row\']: entry for entry in training_catalog(manifest)}\n    selected = list(dict.fromkeys(manifest_rows))\n    if not selected or any(type(i) is not int or i not in catalog for i in selected):\n        raise ValueError(\'Select only manifest row numbers shown in the train catalog\')\n    strengths, seeds = list(strengths), list(seeds)\n    if (type(steps) is not int or steps < 2 or not strengths or not seeds\n            or any(not np.isfinite(s) or not 0 < s < 1 or int(steps*s) < 1 for s in strengths)\n            or any(type(s) is not int or s < 0 for s in seeds)):\n        raise ValueError(\'Need valid strengths, integer seeds and nonempty schedules\')\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    import torch\n    try:\n        diffusers_version = importlib.metadata.version(\'diffusers\')\n    except importlib.metadata.PackageNotFoundError:\n        diffusers_version = None\n    if pipeline is None:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Run generation on a Colab GPU; local CPU generation is not started\')\n        from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n        kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n        if revision:\n            kwargs[\'revision\'] = revision\n        pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n        pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)\n        pipeline.enable_attention_slicing()\n        pipeline.to(\'cuda\')\n    destination.mkdir(parents=True, exist_ok=False)\n    prompt = (\'a single isolated Korean Hangul glyph, black ink, plain white background, \'\n              \'flat front view, clean printed typography\')\n    negative = \'photograph, scene, paper texture, shadows, colored background, decorations\'\n    report = dict(experiment=\'pretrained_img2img_domain_pilot\', free_text_generation=False,\n                  guidance_applied=False, model_id=model_id, requested_revision=revision,\n                  pipeline_class=type(pipeline).__name__,\n                  resolved_commit=getattr(pipeline, \'config\', {}).get(\'_commit_hash\'),\n                  scheduler=dict(pipeline.scheduler.config),\n                  torch_version=torch.__version__, diffusers_version=diffusers_version,\n                  training_manifest=str(Path(manifest).resolve()),\n                  training_manifest_sha256=hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),\n                  training_manifest_checkpoint_link=\'caller must use the actual checkpoint training manifest\',\n                  prompt=prompt, negative_prompt=negative, steps=steps, strengths=strengths,\n                  seeds=seeds, preprocessing=\'fixed 1-luminance; bilinear resize; no crop/threshold/repair\',\n                  results=[])\n\n    def save(image, entry, reference, strokes, label, **metadata):\n        prefix = destination / f\'sample_{len(report["results"]):04d}\'\n        image.save(str(prefix) + \'_raw.png\')\n        ink = scored_ink(image, reference.shape[-1])\n        Image.fromarray(np.uint8(np.clip(1-ink[0], 0, 1)*255)).save(str(prefix)+\'_scored.png\')\n        np.savez_compressed(str(prefix)+\'.npz\', image=ink, reference=reference, reference_strokes=strokes)\n        report[\'results\'].append(dict(**entry, label=label, input=str(prefix)+\'_raw.png\',\n                                     npz=str(prefix)+\'.npz\', **metadata))\n        (destination/\'generation_report.json\').write_text(\n            json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n    for row in selected:\n        entry = catalog[row]\n        reference,strokes=load_catalog_reference(entry)\n        initial = Image.fromarray(np.uint8((1-reference[0])*255)).convert(\'RGB\')\n        save(initial, entry, reference, strokes, \'reference_control\', control=True)\n        initial = initial.resize((512, 512), Image.Resampling.BICUBIC)\n        initial.save(destination/f\'row_{row}_initial_512.png\')\n        for strength in strengths:\n            for seed in seeds:\n                print(f\'Generate train row={row}, strength={strength}, seed={seed}\', flush=True)\n                result = pipeline(prompt=prompt, negative_prompt=negative, image=initial,\n                                  strength=float(strength), num_inference_steps=steps,\n                                  guidance_scale=7.5, generator=torch.Generator(\'cpu\').manual_seed(seed))\n                flags = getattr(result, \'nsfw_content_detected\', None)\n                save(result.images[0], entry, reference, strokes, f\'strength={strength} seed={seed}\',\n                     control=False, strength=float(strength), seed=seed,\n                     safety_filtered=bool(flags[0]) if flags is not None else None)\n    return report\n\n\ndef score_pilot(generation_report, checkpoint, *, score_config=None, source_config=None, show=True):\n    """Load the frozen existing extractor only AFTER the generator is released."""\n    import torch\n    from stroke_model.infer_topology import load_topology\n    from stroke_model.topology_scoring import score_topology_results\n    report = json.loads(Path(generation_report).read_text(encoding=\'utf-8\'))\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model, size, epoch = load_topology(checkpoint, device)\n    extraction = dict(checkpoint=str(checkpoint), checkpoint_epoch=epoch,\n                      output_mode=getattr(model, \'output_mode\', \'independent\'), results=[])\n    root = Path(generation_report).parent / (\'extraction_\'+datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    root.mkdir(exist_ok=False)\n    for index, sample in enumerate(report[\'results\']):\n        with np.load(sample[\'npz\'], allow_pickle=False) as data:\n            arrays = {key: data[key].copy() for key in (\'image\', \'reference\', \'reference_strokes\')}\n        if arrays[\'reference\'].shape != (1, size, size):\n            raise ValueError(\'Training reference resolution differs from checkpoint; do not rescale strokes\')\n        with torch.inference_mode():\n            predicted = model(*[torch.from_numpy(arrays[k])[None].to(device)\n                                for k in (\'image\', \'reference\', \'reference_strokes\')])\n        arrays.update({k: predicted[v][0].float().cpu().numpy()\n                       for k, v in ((\'soft_masks\', \'masks\'), (\'probabilities\', \'probabilities\'))})\n        path = root/f\'sample_{index:04d}.npz\'\n        np.savez_compressed(path, **arrays)\n        extraction[\'results\'].append({**sample, \'npz\': str(path)})\n    del model\n    result = score_topology_results(extraction, score_config=score_config, source_config=source_config,\n                                   save_dir=root/\'scores\', show=show, export_debug=True)\n    import pandas as pd\n    rows = []\n    for sample, measured in zip(extraction[\'results\'], result[\'indicator\'][\'results\']):\n        rows.append({key: sample.get(key) for key in\n                     (\'manifest_row\', \'char_id\', \'label\', \'control\', \'strength\', \'seed\', \'safety_filtered\',\n                      \'page_id\', \'glyph_count\', \'slot\', \'crop_bbox\')}\n                    | {key: measured[key] for key in (\'structural_score\', \'missing_score_cap\',\n                       \'split_score_cap\', \'junction_score_cap\', \'extra_score_cap\')})\n    pd.DataFrame(rows).to_csv(root/\'pilot_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    import zipfile\n    with zipfile.ZipFile(result[\'debug_zip\'], \'a\', zipfile.ZIP_DEFLATED) as archive:\n        archive.write(generation_report, \'generation_report.json\')\n        archive.write(root/\'pilot_scores.csv\', \'pilot_scores.csv\')\n        for index, sample in enumerate(report[\'results\']):\n            raw = Path(sample[\'input\'])\n            archive.write(raw, f\'generated_originals/{index:04d}.png\')\n        for page in report.get(\'pages\', []):\n            archive.write(page[\'input\'], f\'generated_sheets/{page["page_id"]:03d}.png\')\n    return result, pd.DataFrame(rows)\n', 'stroke_model/convert_reference_csv.py': '"""Strict legacy CSV adapter. Ambiguous/missing labels remain unknown (-1)."""\nimport ast\nimport hashlib\nimport json\nimport random\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nfrom tqdm.auto import tqdm\n\n\ndef normalized(value):\n    return unicodedata.normalize(\'NFC\', str(value).replace(\'\\\\\', \'/\'))\n\n\ndef existing_path(value):\n    """Resolve NFC/NFD differences without assuming the shared drive spelling."""\n    path = Path(value)\n    if path.exists():\n        return path\n    current = Path(path.anchor) if path.is_absolute() else Path(\'.\')\n    parts = path.parts[1:] if path.is_absolute() else path.parts\n    for part in parts:\n        direct = current / part\n        if direct.exists():\n            current = direct\n            continue\n        if not current.is_dir():\n            raise FileNotFoundError(value)\n        matches = [p for p in current.iterdir() if normalized(p.name) == normalized(part)]\n        if len(matches) != 1:\n            raise FileNotFoundError(f\'경로를 찾거나 구분할 수 없습니다: {value}\')\n        current = matches[0]\n    return current\n\n\ndef path_list(value):\n    text = str(value).strip()\n    if not text or text.lower() in (\'nan\', \'none\'):\n        return []\n    try:\n        parsed = ast.literal_eval(text)\n        if isinstance(parsed, (list, tuple)):\n            return [str(x) for x in parsed]\n    except (ValueError, SyntaxError):\n        pass\n    for sep in (\';\', \'|\', \'\\n\', \',\'):\n        if sep in text:\n            return [p.strip().strip(\'\\"\\\'\') for p in text.split(sep) if p.strip()]\n    return [text.strip(\'\\"\\\'\')]\n\n\nclass Resolver:\n    def __init__(self, csv_path, dataset_root, local_only=False):\n        self.csv_path, self.root = csv_path, dataset_root\n        self.local_only = local_only\n        self.index = None\n\n    def __call__(self, value):\n        raw = normalized(str(value).strip().strip(\'\\"\\\'\'))\n        candidates = (Path(raw), self.csv_path.parent / raw, self.root / raw)\n        if self.local_only:\n            candidate = (self.root / raw).resolve()\n            candidates = (candidate,) if candidate.is_relative_to(self.root.resolve()) else ()\n        for candidate in candidates:\n            try:\n                found = existing_path(candidate)\n                if found.is_file():\n                    return found\n            except FileNotFoundError:\n                pass\n        if self.index is None:\n            self.index = {}\n            for p in self.root.rglob(\'*\'):\n                if \'input_stroke_ga\' in p.relative_to(self.root).parts:\n                    continue  # Legacy config excludes this duplicate, unused folder.\n                if p.is_file() and p.suffix.lower() in (\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'):\n                    self.index.setdefault(normalized(p.name), []).append(p)\n        candidates = self.index.get(raw.split(\'/\')[-1], [])\n        if not candidates:\n            raise FileNotFoundError(f\'CSV 이미지가 없습니다: {value} (검색 범위: {self.root})\')\n        def suffix_score(p):\n            score = 0\n            for left, right in zip(reversed(normalized(p).split(\'/\')), reversed(raw.split(\'/\'))):\n                if left != right:\n                    break\n                score += 1\n            return score\n        best = max(map(suffix_score, candidates))\n        matches = [p for p in candidates if suffix_score(p) == best]\n        if len(matches) != 1:\n            raise ValueError(f\'같은 파일명이 여러 곳에 있습니다. CSV 경로를 명확하게 해주세요: {value}: {matches}\')\n        return matches[0]\n\n\ndef load_ink(path, size):\n    with Image.open(path) as source:\n        # Composite transparent PNGs onto white before converting to grayscale.\n        image = ImageOps.exif_transpose(source).convert(\'RGBA\')\n        background = Image.new(\'RGBA\', image.size, (255, 255, 255, 255))\n        image = Image.alpha_composite(background, image).convert(\'L\')\n        x = np.asarray(image.resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32) / 255\n    if x.mean() > .5:\n        x = 1 - x\n    return x[None].copy()\n\n\ndef convert_csv(csv_path, output, size=128, seed=42, dataset_root=None, split=None, local_only=False,\n                independent=False):\n    if split not in (None, \'test\'):\n        raise ValueError(\'split must be None (train/val) or test\')\n    csv_path = existing_path(csv_path)\n    root = existing_path(dataset_root) if dataset_root else csv_path.parent\n    resolve = Resolver(csv_path, root, local_only=local_only)\n    df = pd.read_csv(csv_path)\n    names = {str(c).lower(): c for c in df.columns}\n    def column(*aliases, optional=False):\n        for name in aliases:\n            if name.lower() in names:\n                return names[name.lower()]\n        if optional:\n            return None\n        raise ValueError(f\'필수 컬럼 {aliases} 없음. 현재 CSV 컬럼: {list(df.columns)}\')\n    cols = {\n        \'image\': column(\'I_prep\', \'input_image_path\', \'input_path\', \'input_image\', \'input\', \'source_path\'),\n        \'reference\': column(\'I_g\', \'reference_image\', \'reference_path\', \'target_image\', \'target_path\', \'glyph_path\'),\n        \'strokes\': column(\'I_G_strokes\', \'reference_strokes\', \'reference_stroke_paths\', \'template_strokes\', \'target_strokes\', \'target_stroke_paths\'),\n        \'targets\': column(\'input_stroke_paths\', \'input_strokes\', \'input_strokes_paths\', \'prep_stroke_paths\', \'gt_stroke_paths\', \'gt_strokes\', \'answer_stroke_paths\', \'stroke_paths\'),\n    }\n    group_col = column(\'writer_id\', \'writer\', \'subject_id\', \'group_id\', optional=True)\n    char_col = column(\'char_id\', \'char\', \'character\', \'text\', optional=True)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    (output / \'samples\').mkdir()\n    rows, reports, skipped_rows = [], [], []\n    seen_images = {}\n    for index, row in tqdm(df.iterrows(), total=len(df), desc=\'Convert images to NPZ\', unit=\'sample\'):\n        try:\n            image_path = resolve(row[cols[\'image\']])\n            image = load_ink(image_path, size)\n            reference = load_ink(resolve(row[cols[\'reference\']]), size)\n            ref_paths, target_paths = path_list(row[cols[\'strokes\']]), path_list(row[cols[\'targets\']])\n            if not ref_paths or len(ref_paths) != len(target_paths):\n                raise ValueError(f\'기준/정답 획 수 불일치: {len(ref_paths)} / {len(target_paths)}\')\n            strokes = np.concatenate([load_ink(resolve(p), size) for p in ref_paths])\n            targets = np.concatenate([load_ink(resolve(p), size) for p in target_paths]) > .5\n            empty_reference = strokes.reshape(len(strokes), -1).max(1) <= .5\n            if empty_reference.any():\n                empty_paths = [p for p, empty in zip(ref_paths, empty_reference) if empty]\n                if independent:\n                    skipped_rows.append({\'csv_row\': int(index) + 1,\n                        \'input_path\': str(row[cols[\'image\']]),\n                        \'empty_reference_paths\': \';\'.join(empty_paths),\n                        \'reason\': \'empty_reference_stroke_after_resize\'})\n                    if len(skipped_rows) <= 5:\n                        print(f\'건너뜀: CSV {index + 1}번째 행, 빈 기준 획: {empty_paths}\', flush=True)\n                    continue\n                raise ValueError(f\'비어 있는 기준 획 이미지가 있습니다: {empty_paths}\')\n            foreground = image[0] > .5\n            counts = targets.sum(0)\n            known = foreground & (counts == 1)\n            labels = np.full((size, size), -1, dtype=np.int64)\n            labels[known] = targets.argmax(0)[known]\n            if not independent and any(not (labels == k).any() for k in range(len(strokes))):\n                raise ValueError(\'정답 획 중 단독 소유 픽셀이 없는 획이 있습니다. 획 마스크/극성을 확인해주세요\')\n            digest = hashlib.sha256(image.tobytes()).hexdigest()\n            if group_col:\n                if pd.isna(row[group_col]):\n                    raise ValueError(\'필자/그룹 ID가 비어 있습니다\')\n                group = \'writer:\' + str(row[group_col])\n            elif independent:\n                # Keep augmented variants sharing a font/character reference together.\n                group = \'reference:\' + hashlib.sha256(reference.tobytes()).hexdigest()\n            else:\n                group = \'image:\' + digest\n            if digest in seen_images and seen_images[digest] != group:\n                raise ValueError(\'동일 입력 이미지에 서로 다른 필자/그룹 ID가 지정되었습니다\')\n            seen_images[digest] = group\n            path = output / \'samples\' / f\'{index:05d}.npz\'\n            arrays = dict(image=image, reference=reference, strokes=strokes, labels=labels)\n            if independent:\n                # Actual input annotations, including shared crossings and empty slots.\n                # These are NOT reconstructed from exclusive labels or templates.\n                arrays.update(target_masks=targets.astype(np.float32),\n                              valid_pixels=np.ones_like(image))\n            np.savez_compressed(path, **arrays)\n            coverage = float(known.sum() / max(foreground.sum(), 1))\n            rows.append({\'path\': str(path.resolve()), \'group_id\': group, \'source\': \'legacy_csv_unreviewed\',\n                         \'char_id\': str(row[char_col]) if char_col else \'\', \'source_image\': str(image_path)})\n            reports.append({\'row\': int(index), \'foreground_pixels\': int(foreground.sum()),\n                            \'annotation_coverage\': coverage, \'overlap_pixels\': int((foreground & (counts > 1)).sum()),\n                            \'missing_pixels\': int((foreground & (counts == 0)).sum())})\n        except (ValueError, FileNotFoundError) as exc:\n            raise ValueError(f\'CSV 데이터 {index + 1}번째 행: {exc}\') from exc\n    pd.DataFrame(skipped_rows, columns=[\'csv_row\', \'input_path\', \'empty_reference_paths\', \'reason\']).to_csv(\n        output / \'skipped_samples.csv\', index=False, encoding=\'utf-8-sig\')\n    groups = sorted({r[\'group_id\'] for r in rows})\n    if not rows:\n        raise ValueError(\'CSV에 데이터가 없습니다\')\n    if split is None and len(groups) < 2:\n        raise ValueError(\'검증 분리를 위해 최소 2개 필자/원본 그룹이 필요합니다\')\n    random.Random(seed).shuffle(groups)\n    validation = set(groups[:max(1, round(len(groups) * .2))])\n    for row in rows:\n        row[\'split\'] = split or (\'val\' if row[\'group_id\'] in validation else \'train\')\n    manifest = output / \'manifest.json\'\n    manifest.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    report = {\'csv\': str(csv_path), \'columns\': cols, \'group_column\': group_col,\n              \'split_note\': \'전체 test 전용, 학습/증강 없음\' if split == \'test\' else (\n                  \'필자 그룹 분할\' if group_col else (\n                      \'기준 이미지 그룹 분할: 같은 기준의 합성 변형은 같은 split; 필자 일반화 평가는 아님\'\n                      if independent else \'원본 이미지 분할: 필자 일반화 평가는 아님\')),\n              \'label_policy\': (\'실제 입력 획별 이진 마스크 전체 감독; 교차점 공유/빈 획 보존. CSV 획 순서 대응 가정.\'\n                               if independent else \'단독 소유 전경만 감독; 교차/미할당은 -1. CSV 획 순서가 대응한다고 가정.\'),\n              \'supervision\': \'actual_input_multihot\' if independent else \'exclusive_annotated\',\n              \'skipped_count\': len(skipped_rows), \'skipped_rows\': skipped_rows,\n              \'rows\': reports}\n    (output / \'conversion_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(f"변환 완료: train={sum(r[\'split\']==\'train\' for r in rows)}, val={sum(r[\'split\']==\'val\' for r in rows)}, test={sum(r[\'split\']==\'test\' for r in rows)}")\n    print(report[\'split_note\'])\n    print(report[\'label_policy\'])\n    print(f\'빈 기준 획으로 제외한 샘플: {len(skipped_rows)}개 (skipped_samples.csv)\')\n    print(\'획 순서의 의미적 대응은 자동 검증되지 않습니다. conversion_report.json과 미리보기를 확인하세요.\')\n    return manifest\n', 'stroke_model/prepare_stroke_zips.py': '"""Stage the four stroke archives locally; use CSV for semantic correspondence."""\nimport hashlib\nimport json\nimport shutil\nimport stat\nimport tarfile\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nfrom tqdm.auto import tqdm\n\nfrom .convert_reference_csv import convert_csv, existing_path\n\nARCHIVES = (\'input_stroke.zip\', \'input.zip\', \'target_stroke.zip\', \'target.zip\')\n\n\ndef discover_archives(source, roles=ARCHIVES):\n    """Discover every supported archive recursively, regardless of name/count."""\n    paths = []\n    for name in roles:\n        role = Path(name).stem\n        try:\n            folder = existing_path(source / role)\n        except FileNotFoundError:\n            folder = source / role\n        parts = []\n        if folder.is_dir():\n            extensions = (\'.tar\', \'.tar.gz\', \'.tgz\', \'.tar.bz2\', \'.tbz2\', \'.tar.xz\', \'.txz\', \'.zip\')\n            parts = sorted((path for path in folder.rglob(\'*\')\n                            if path.is_file() and path.name.casefold().endswith(extensions)),\n                           key=lambda path: path.relative_to(folder).as_posix().casefold())\n        if parts:\n            paths.extend(parts)\n        else:\n            try:\n                paths.append(existing_path(source / name))\n            except FileNotFoundError as exc:\n                raise FileNotFoundError(\n                    f\'{source / role} 안의 TAR/ZIP 압축 파일 \'\n                    f\'또는 {source / name}을 찾을 수 없습니다.\') from exc\n    return paths\n\n\ndef extract_checked(archive, destination):\n    destination = Path(destination).resolve()\n    if not zipfile.is_zipfile(archive):\n        with tarfile.open(archive, \'r:*\') as tar:\n            members = tar.getmembers()\n            for member in members:\n                path = PurePosixPath(member.name.replace(\'\\\\\', \'/\'))\n                if (path.is_absolute() or \'..\' in path.parts or\n                        any(\':\' in part for part in path.parts) or\n                        not (member.isfile() or member.isdir())):\n                    raise ValueError(f\'Unsafe TAR member: {member.name}\')\n            needed = sum(m.size for m in members if m.isfile())\n            if needed > shutil.disk_usage(destination.parent).free:\n                raise OSError(f\'로컬 디스크 부족: 압축 해제에 {needed / 2**30:.1f} GiB 필요\')\n            for member in tqdm(members, desc=f\'Extract {Path(archive).name}\', unit=\'file\', mininterval=2):\n                target = destination.joinpath(*PurePosixPath(member.name.replace(\'\\\\\', \'/\')).parts)\n                if member.isdir():\n                    target.mkdir(parents=True, exist_ok=True)\n                else:\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with tar.extractfile(member) as src, target.open(\'xb\') as dst:\n                        shutil.copyfileobj(src, dst)\n        return\n    with zipfile.ZipFile(archive) as z:\n        members = z.infolist()\n        for member in members:\n            path = PurePosixPath(member.filename.replace(\'\\\\\', \'/\'))\n            if (path.is_absolute() or \'..\' in path.parts or\n                    any(\':\' in part for part in path.parts) or\n                    stat.S_ISLNK(member.external_attr >> 16)):\n                raise ValueError(f\'Unsafe ZIP member: {member.filename}\')\n        needed = sum(m.file_size for m in members)\n        if needed > shutil.disk_usage(destination.parent).free:\n            raise OSError(f\'로컬 디스크 부족: 압축 해제에 {needed / 2**30:.1f} GiB 필요\')\n        for member in tqdm(members, desc=f\'Extract {Path(archive).name}\', unit=\'file\', mininterval=2):\n            z.extract(member, destination)\n\n\ndef find_csv(source, raw, csv_path=None):\n    if csv_path:\n        return existing_path(csv_path)\n    # Only the selected dataset directory and extracted archives, never a test/old dataset.\n    direct = [p for p in source.iterdir() if p.is_file() and p.name.casefold() in (\'path.csv\', \'paths.csv\')]\n    candidates = direct or [p for p in raw.rglob(\'*\') if p.is_file() and p.name.casefold() in (\'path.csv\', \'paths.csv\')]\n    if not candidates:\n        raise FileNotFoundError(\n            f\'path.csv / paths.csv가 아직 없습니다. {source}에 업로드한 뒤 이 셀만 다시 실행하세요. \'\n            \'완료된 압축 파일 복사/해제는 재사용합니다. 다른 CSV 이름이면 CSV_PATH를 지정하세요.\')\n    if len(candidates) != 1:\n        raise ValueError(f\'CSV 후보가 여러 개입니다. CSV_PATH를 지정하세요: {candidates}\')\n    return candidates[0]\n\n\ndef stage_stroke_archives(source, local_root=\'/content/hsqm_stroke_zip\', roles=None):\n    """Share the training archive cache while staging only requested roles."""\n    source = existing_path(source)\n    paths = discover_archives(source)\n    signature = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in paths]\n    key = hashlib.sha256(b\'archive_layout2:\' + json.dumps(signature).encode()).hexdigest()[:16]\n    cache = Path(local_root) / key\n    raw = cache / \'raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    allowed = set(discover_archives(source, roles=roles)) if roles is not None else set(paths)\n    print(\'사용할 압축 파일:\',len(allowed),\'개 / 전체\',len(paths),\'개\',flush=True)\n    for path in paths:\n        if path not in allowed:\n            continue\n        # Different folders may contain archives with identical filenames.\n        archive_id = hashlib.sha256(path.relative_to(source).as_posix().encode()).hexdigest()[:16]\n        destination = raw / archive_id\n        if (destination / \'.complete.json\').is_file():\n            print(\'로컬 압축 데이터 재사용:\', path.name, flush=True)\n            continue\n        # Failed copies/extractions never create the completion marker.\n        local_zip = cache / (archive_id + \'_\' + path.name)\n        before = (path.stat().st_size, path.stat().st_mtime_ns)\n        if shutil.disk_usage(cache).free < before[0]:\n            raise OSError(\'압축 파일 복사를 위한 로컬 디스크 공간이 부족합니다\')\n        print(\'Drive → 로컬 압축 파일 복사:\', path.name, flush=True)\n        partial = local_zip.with_suffix(\'.partial\')\n        with path.open(\'rb\') as src, partial.open(\'wb\') as dst:\n            with tqdm(total=before[0], desc=path.name, unit=\'B\', unit_scale=True, mininterval=2) as progress:\n                while chunk := src.read(8 * 1024 * 1024):\n                    dst.write(chunk)\n                    progress.update(len(chunk))\n        after = (path.stat().st_size, path.stat().st_mtime_ns)\n        if before != after or partial.stat().st_size != before[0]:\n            raise RuntimeError(f\'{path.name}이 복사 중 변경됐습니다. 업로드 완료 후 재실행하세요.\')\n        partial.replace(local_zip)\n        temporary = Path(tempfile.mkdtemp(prefix=path.stem + \'_\', dir=cache))\n        extract_checked(local_zip, temporary)\n        (temporary / \'.complete.json\').write_text(json.dumps(before), encoding=\'utf-8\')\n        temporary.rename(destination)\n        local_zip.unlink()  # Keep extracted images, reclaim the local archive space.\n    return cache, raw, signature\n\n\ndef prepare_stroke_zips(source, local_root=\'/content/hsqm_stroke_zip\', csv_path=None, size=128, seed=42,\n                        independent=False):\n    source = existing_path(source)\n    cache, raw, signature = stage_stroke_archives(source, local_root)\n    selected = find_csv(source, raw, csv_path)\n    content = selected.read_bytes()\n    conversion_key = hashlib.sha256(content + f\'converter3:{size}:{seed}:{independent}\'.encode()).hexdigest()[:16]\n    prepared = cache / (\'prepared_\' + conversion_key)\n    manifest = prepared / \'manifest.json\'\n    if (prepared / \'.complete.json\').is_file() and manifest.is_file():\n        print(\'로컬 변환 데이터 재사용:\', manifest, flush=True)\n        return manifest\n    print(\'CSV 선택:\', selected, flush=True)\n    local_csv = cache / \'selected_paths.csv\'\n    local_csv.write_bytes(content)\n    temporary = Path(tempfile.mkdtemp(prefix=\'conversion_\', dir=cache)) / \'prepared\'\n    manifest = convert_csv(local_csv, temporary, size=size, seed=seed, dataset_root=raw, local_only=True,\n                           independent=independent)\n    # Keep converter absolute NPZ paths stable: use the completed temp directory directly.\n    prepared.mkdir(exist_ok=True)\n    rows = json.loads(manifest.read_text(encoding=\'utf-8\'))\n    (prepared / \'manifest.json\').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    shutil.copyfile(temporary / \'conversion_report.json\', prepared / \'conversion_report.json\')\n    shutil.copyfile(temporary / \'skipped_samples.csv\', prepared / \'skipped_samples.csv\')\n    (prepared / \'selected_paths.csv\').write_bytes(content)\n    (prepared / \'.complete.json\').write_text(json.dumps({\'csv_source\': str(selected), \'archives\': signature}), encoding=\'utf-8\')\n    return prepared / \'manifest.json\'\n', 'stroke_model/reference_only_data.py': '"""Restore training-reference membership without converting training samples."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\n\nimport numpy as np\nimport pandas as pd\nfrom tqdm.auto import tqdm\n\nfrom .convert_reference_csv import Resolver, load_ink, normalized, path_list\nfrom .prepare_stroke_zips import stage_stroke_archives, discover_archives\n\nREFERENCE_ROLES = (\'target.zip\', \'target_stroke.zip\')\n\n\ndef prepare_reference_catalog(source, csv_path, conversion, size, seed, local_root):\n    """Reproduce the original group split from the saved accepted CSV row IDs.\n\n    Only reference PNGs are decoded to recover content-hash reference groups.\n    Ordered stroke PNGs are opened later, when a glyph is selected. No NPZ here.\n    """\n    source, csv_path = Path(source), Path(csv_path)\n    group_col = conversion.get(\'group_column\')\n    if conversion.get(\'supervision\') != \'actual_input_multihot\' and not group_col:\n        raise ValueError(\'이전 image-hash 분할은 기준만으로 복원할 수 없습니다. \'\n                         \'REFERENCE_ONLY=False로 기존 전체 변환을 사용하세요.\')\n    if \'rows\' not in conversion:\n        raise ValueError(\'저장된 conversion_report.json에 원래 학습 행 목록이 필요합니다.\')\n    cache, raw, _ = stage_stroke_archives(source, local_root, roles=REFERENCE_ROLES)\n    fingerprint = hashlib.sha256(csv_path.read_bytes()+json.dumps(\n        dict(version=1,size=size,seed=seed,conversion=conversion),sort_keys=True).encode()).hexdigest()[:20]\n    destination = cache/f\'reference_only_{fingerprint}\'\n    manifest = destination/\'manifest.json\'\n    if manifest.is_file():\n        rows=json.loads(manifest.read_text(encoding=\'utf-8\'))\n        if rows and all(Path(r[\'reference\']).is_file() and all(Path(p).is_file() for p in r[\'reference_strokes\']) for r in rows):\n            print(\'기준 글자 목록 캐시 재사용 (학습 NPZ 변환 없음):\',manifest,flush=True)\n            return manifest\n    frame=pd.read_csv(csv_path)\n    columns=conversion[\'columns\']\n    accepted=[int(row[\'row\']) for row in conversion[\'rows\']]\n    if not accepted or len(set(accepted))!=len(accepted) or min(accepted)<0 or max(accepted)>=len(frame):\n        raise ValueError(\'저장된 학습 행과 CSV가 일치하지 않습니다.\')\n    resolve=Resolver(csv_path,raw,local_only=True)\n    # Index ONLY target archives, even if input archives were extracted earlier.\n    resolve.index={}\n    for archive in discover_archives(source,roles=REFERENCE_ROLES):\n        archive_id=hashlib.sha256(archive.relative_to(source).as_posix().encode()).hexdigest()[:16]\n        for path in (raw/archive_id).rglob(\'*\'):\n            if path.is_file() and path.suffix.lower() in (\'.png\',\'.jpg\',\'.jpeg\',\'.bmp\',\'.webp\',\'.tif\',\'.tiff\'):\n                resolve.index.setdefault(normalized(path.name),[]).append(path)\n    char_col=next((c for c in frame.columns if str(c).lower() in (\'char_id\',\'char\',\'character\',\'text\')),None)\n    hashes, resolved, records, groups = {}, {}, [], set()\n    seen=set()\n    print(\'학습 입력/정답 획은 읽지 않고 기준 글자만 준비합니다.\',flush=True)\n    for index in tqdm(accepted,desc=\'Index reference glyphs (no NPZ)\',unit=\'row\',mininterval=2):\n        row=frame.iloc[index]\n        raw_reference=str(row[columns[\'reference\']])\n        if raw_reference not in resolved:\n            resolved[raw_reference]=resolve(raw_reference)\n        reference=resolved[raw_reference]\n        if str(reference) not in hashes:\n            hashes[str(reference)]=hashlib.sha256(load_ink(reference,size).tobytes()).hexdigest()\n        if group_col:\n            if pd.isna(row[group_col]):raise ValueError(\'학습 그룹 ID가 비어 있습니다.\')\n            group=\'writer:\'+str(row[group_col])\n        else:\n            group=\'reference:\'+hashes[str(reference)]\n        groups.add(group)\n        stroke_names=path_list(row[columns[\'strokes\']])\n        if not stroke_names:raise ValueError(\'기준 획 목록이 비어 있습니다.\')\n        key=(str(reference),tuple(stroke_names),group)\n        if key in seen:continue\n        seen.add(key)\n        paths=[]\n        for name in stroke_names:\n            if name not in resolved:resolved[name]=resolve(name)\n            paths.append(str(resolved[name].resolve()))\n        signature=hashlib.sha256((hashes[str(reference)]+json.dumps(paths)).encode()).hexdigest()\n        records.append(dict(reference=str(reference.resolve()),reference_strokes=paths,\n            image_size=size,group_id=group,char_id=str(row[char_col]) if char_col else \'\',\n            source_csv_row=index+1,reference_sha256=signature,format=\'reference_paths_v1\'))\n    groups=sorted(groups)\n    if len(groups)<2:raise ValueError(\'원래 학습 분할을 복원할 그룹이 부족합니다.\')\n    random.Random(seed).shuffle(groups)\n    validation=set(groups[:max(1,round(len(groups)*.2))])\n    for record in records:record[\'split\']=\'val\' if record[\'group_id\'] in validation else \'train\'\n    destination.mkdir(exist_ok=True)\n    partial=manifest.with_suffix(\'.partial\')\n    partial.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    partial.replace(manifest)\n    print(f\'기준 {len(records)}개 준비 완료. 학습 표본 {len(accepted)}개의 NPZ 변환을 생략했습니다.\',flush=True)\n    return manifest\n', 'repair_audit/selectable_target_inputs.py': '"""Freeze four user-selected reference glyphs without any prior experiment ZIP."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\n\nfrom diffusion_model.score_pilot import load_catalog_reference\nfrom .fresh_target_inputs import canonical_sheet\nfrom .stage1 import json_write\n\n\ndef load_reference_image(entry):\n    """Read generation appearance only, never that font\'s stroke annotations."""\n    if entry.get(\'path\'):\n        with np.load(entry[\'path\'],allow_pickle=False) as data: reference=data[\'reference\'].astype(np.float32)\n    else:\n        from stroke_model.convert_reference_csv import load_ink\n        reference=load_ink(entry[\'reference\'],entry[\'image_size\'])\n    if (reference.ndim!=3 or reference.shape[0]!=1 or reference.shape[1]!=reference.shape[2]\n        or not np.isfinite(reference).all() or reference.min()<0 or reference.max()>1 or reference.max()<=.5):\n        raise ValueError(\'Invalid generation reference image\')\n    return reference\n\n\ndef _signature(reference,strokes):\n    digest=hashlib.sha256()\n    for array in (reference,strokes):\n        array=np.ascontiguousarray(array,dtype=np.float32)\n        digest.update(str(array.shape).encode());digest.update(array.tobytes())\n    return digest.hexdigest()\n\n\ndef load_prototype(bank,key):\n    root=Path(bank);index=json.loads((root/\'index.json\').read_text(encoding=\'utf-8\'))\n    if key not in index: raise ValueError(f\'No locked prototype for character: {key}\')\n    record=index[key]\n    # Filename derives from the key, not from paths in the index.\n    name=hashlib.sha256(key.encode()).hexdigest()+\'.npz\'\n    with np.load(root/name,allow_pickle=False) as data:\n        reference=data[\'reference\'].copy();strokes=data[\'strokes\'].copy()\n    if _signature(reference,strokes)!=record[\'sha256\']:raise ValueError(\'Locked prototype content changed\')\n    return dict(reference=reference,strokes=strokes),record\n\n\ndef lock_prototype(catalog,row,key,bank):\n    """Pin one immutable template per character; no font-dependent lookup."""\n    key=str(key).strip()\n    if not key:raise ValueError(\'An explicit character identity is required\')\n    entry=next((e for e in catalog if e[\'manifest_row\']==row),None)\n    if entry is None:raise ValueError(\'Unknown prototype row\')\n    reference,strokes=load_catalog_reference(entry)\n    signature=_signature(reference,strokes)\n    root=Path(bank);index_path=root/\'index.json\'\n    index=json.loads(index_path.read_text(encoding=\'utf-8\')) if index_path.is_file() else {}\n    if key in index:\n        _,record=load_prototype(root,key)\n        if record[\'sha256\']!=signature:\n            raise ValueError(f\'{key}: prototype already locked; cannot switch it with the input font\')\n        return record\n    root.mkdir(parents=True,exist_ok=True)\n    record=dict(character=key,manifest_row=row,sha256=signature,reference=entry)\n    np.savez_compressed(root/(hashlib.sha256(key.encode()).hexdigest()+\'.npz\'),reference=reference,strokes=strokes)\n    index[key]=record\n    partial=root/\'index.partial.json\';json_write(partial,index);partial.replace(index_path)\n    return record\n\n\ndef save_prototype(catalog,row,key,bank):\n    """Editable experiment setup. Existing experiment NPZ snapshots stay intact."""\n    key=str(key).strip()\n    if not key:raise ValueError(\'글자 이름을 먼저 입력하세요.\')\n    entry=next((e for e in catalog if e[\'manifest_row\']==row),None)\n    if entry is None:raise ValueError(\'Unknown prototype row\')\n    reference,strokes=load_catalog_reference(entry)\n    root=Path(bank);root.mkdir(parents=True,exist_ok=True)\n    index_path=root/\'index.json\'\n    index=json.loads(index_path.read_text(encoding=\'utf-8\')) if index_path.is_file() else {}\n    record=dict(character=key,manifest_row=row,sha256=_signature(reference,strokes),reference=entry)\n    name=hashlib.sha256(key.encode()).hexdigest()+\'.npz\'\n    partial_data=root/(name+\'.partial\')\n    with partial_data.open(\'wb\') as stream:np.savez_compressed(stream,reference=reference,strokes=strokes)\n    partial_data.replace(root/name)\n    index[key]=record\n    partial_index=root/\'index.partial.json\';json_write(partial_index,index);partial_index.replace(index_path)\n    return record\n\n\ndef prepare_selected_sheet(catalog, rows, output, *, prototype_keys, prototype_bank, size=576):\n    rows=list(rows)\n    by_row={entry[\'manifest_row\']:entry for entry in catalog}\n    if len(rows)!=4 or any(type(row) is not int or row not in by_row for row in rows):\n        raise ValueError(\'Choose four available reference rows, in slot order\')\n    entries=[by_row[row] for row in rows]\n    keys=[str(k).strip() for k in prototype_keys]\n    if len(keys)!=4 or any(not k for k in keys):raise ValueError(\'Need four explicit character identities\')\n    # Input images and repair references are deliberately independent.\n    source_references=[load_reference_image(entry) for entry in entries]\n    loaded=[load_prototype(prototype_bank,key) for key in keys]\n    conditions=[condition for condition,record in loaded]\n    records=[record for condition,record in loaded]\n    if len({c[\'reference\'].shape for c in conditions})!=1:\n        raise ValueError(\'Selected references have different resolutions\')\n    sheet=canonical_sheet([dict(reference=reference) for reference in source_references],size)\n    root=Path(output);root.mkdir(parents=True,exist_ok=False)\n    directory=root/\'group_000_normal_control\';directory.mkdir()\n    sheet.save(directory/\'input.png\')\n    sheet.save(directory/\'generation_reference_sheet.png\')\n    for i,condition in enumerate(conditions,1):\n        np.savez_compressed(directory/f\'condition_{i}.npz\',**condition)\n    case=dict(case_id=\'group_000_normal_control\',kind=\'normal_control\',directory=str(directory),\n        original_rows=rows,char_ids=[str(e.get(\'char_id\',\'\')) for e in entries],\n        selected_references=entries,prototype_keys=keys,prototype_records=records,\n        source=\'user_selected_images_with_locked_character_prototypes\')\n    json_write(directory/\'input_metadata.json\',case)\n    json_write(root/\'selection.json\',dict(slot_order=\'top-left, top-right, bottom-left, bottom-right\',\n        selected_rows=rows,references=entries,canvas_size=size,prototype_keys=keys,prototypes=records,\n        identity_matching=\'user-confirmed character keys; never inferred from font-specific row IDs\',\n        source_matches_prototype=[bool(np.array_equal(a,b[\'reference\'])) for a,b in zip(source_references,conditions)]))\n    return [case]\n', 'repair_audit/selection_cache.py': '"""One-time local image/stroke snapshots and thumbnails for the picker."""\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nfrom pathlib import Path\nimport tempfile\n\nimport numpy as np\nfrom PIL import Image\nfrom diffusion_model.score_pilot import load_catalog_reference\n\n\ndef reference_identity(entry):\n    """Read explicit Hangul metadata/filename characters; never decode numeric IDs."""\n    characters={};current=entry;seen=set()\n    while isinstance(current,dict) and id(current) not in seen:\n        seen.add(id(current))\n        for field in (\'char_id\',\'char\',\'character\',\'glyph\',\'text\'):\n            value=unicodedata.normalize(\'NFC\',str(current.get(field,\'\')).strip())\n            if len(value)==1 and \'\\uac00\'<=value<=\'\\ud7a3\':\n                characters.setdefault(value,[]).append(\'metadata:\'+field)\n        for field in (\'reference\',\'source_image\',\'target_path\'):\n            value=str(current.get(field,\'\'))\n            stem=Path(value.replace(\'\\\\\',\'/\')).stem\n            tokens=re.findall(r\'(?<![가-힣])[가-힣](?![가-힣])\',unicodedata.normalize(\'NFC\',stem))\n            if len(tokens)==1:characters.setdefault(tokens[0],[]).append(field+\':\'+value)\n        current=current.get(\'source_entry\')\n    character=next(iter(characters)) if len(characters)==1 else None\n    return dict(resolved_character=character,character_evidence=characters,\n                character_identity_conflict=len(characters)>1)\n\n\ndef thumbnail(reference,size=92):\n    image=Image.fromarray(np.uint8(np.clip(1-reference[0],0,1)*255)).resize((size,size))\n    buffer=io.BytesIO();image.save(buffer,format=\'PNG\');return buffer.getvalue()\n\n\ndef prepare_selection_cache(catalog,root,*,source_fingerprint,progress=None):\n    """Read each source once; interactions later use only local/in-memory data.\n\n    source_fingerprint identifies an explicit data snapshot. Change it when the\n    original data change. No model, inference, generation, or normalization change.\n    """\n    if not catalog:raise ValueError(\'Empty selection catalog\')\n    signature=hashlib.sha256(json.dumps(dict(version=2,catalog=catalog,\n        source_fingerprint=source_fingerprint),sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    destination=root/signature;index=destination/\'catalog.json\'\n    if index.is_file():\n        cached=json.loads(index.read_text(encoding=\'utf-8\'))\n        if all(Path(e[\'path\']).is_file() and Path(e[\'thumbnail\']).is_file() for e in cached):return cached\n        raise ValueError(\'Incomplete local selection cache; use a new cache directory\')\n    with tempfile.TemporaryDirectory(prefix=\'preparing_\',dir=root) as tmp:\n        stage=Path(tmp);cached=[]\n        for i,entry in enumerate(catalog,1):\n            reference,strokes=load_catalog_reference(entry)\n            name=f"row_{entry[\'manifest_row\']}"\n            np.savez_compressed(stage/(name+\'.npz\'),reference=reference,strokes=strokes)\n            (stage/(name+\'.png\')).write_bytes(thumbnail(reference))\n            identity=reference_identity(entry)\n            cached.append(dict(entry,**identity,char_id=identity[\'resolved_character\'] or entry.get(\'char_id\',\'\'),\n                path=str(destination/(name+\'.npz\')),\n                thumbnail=str(destination/(name+\'.png\')),source_entry=entry,\n                cached_array_sha256=hashlib.sha256(reference.tobytes()+strokes.tobytes()).hexdigest()))\n            if progress:progress(i,len(catalog))\n        (stage/\'catalog.json\').write_text(json.dumps(cached,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n        # Keep the prepared directory, not a partially written cache.\n        stage.rename(destination)\n    return cached\n\n\ndef selection_thumbnails(catalog):\n    return {e[\'manifest_row\']:Path(e[\'thumbnail\']).read_bytes() for e in catalog}\n', 'stroke_model/ink_target_guide.py': '"""Project the aligned glyph silhouette inside a local ink mask."""\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import binary_dilation, distance_transform_edt\nimport torch\n\n\ndef finish_ink_rgb(rgb, source, mask, background=None):\n    if background is None:\n        clean = rgb.mean(1, keepdim=True).expand_as(rgb)\n    else:\n        # Recover the black ink opacity along the known background RGB ray.\n        energy = background.square().sum(1, keepdim=True)\n        transmission = ((rgb*background).sum(1, keepdim=True)/energy.clamp_min(1e-6)).clamp(0,1)\n        clean = background*transmission\n    return torch.where(mask, clean, source)\n\n\nclass InkTargetGuide:\n    def __init__(self, aligned_guide, rendered, background_image=None):\n        self.source = aligned_guide.source.copy()\n        self.target = self.source.copy()\n        self.background = (np.asarray(background_image.convert(\'RGB\'),np.float32)/255\n                           if background_image is not None else None)\n        if self.background is not None and self.background.shape!=self.source.shape:\n            raise ValueError(\'Background must match the sheet size\')\n        self.roi = np.zeros(self.source.shape[:2], bool)\n        self.metadata = dict(kind=\'glyph_ink_projection\', final_target_replacement=False,\n                             outside_roi=\'original_pixels\', inside_roi=\'grayscale_output\')\n        if self.background is not None:\n            self.metadata[\'inside_roi\']=\'black_ink_over_known_background\'\n        self.report = dict(aligned_guide.report, ink_projection=self.metadata)\n        for glyph, strokes in zip(aligned_guide.report[\'glyphs\'], rendered, strict=True):\n            x0, y0, x1, y1 = glyph[\'bbox\']\n            region = np.s_[y0:y1, x0:x1]\n            crop = aligned_guide.source[region]\n            gray = crop @ np.array([.299, .587, .114], np.float32)\n            light = float(np.percentile(gray, 90))\n            if light < .65:\n                raise ValueError(\'글자 마스크 가이드는 밝은 배경의 어두운 글자용입니다.\')\n            alpha = np.asarray(Image.fromarray(strokes[\'aligned_strokes\'].max(0)).resize(\n                (x1-x0, y1-y0), Image.Resampling.BILINEAR)).clip(0, 1)\n            # Exclude the faint edge of the old ink from background samples.\n            old_ink = gray < light - .025\n            excluded = binary_dilation(old_ink, iterations=2)\n            if excluded.all():\n                raise ValueError(\'글자 주변에서 배경 샘플을 찾지 못했습니다.\')\n            indices = distance_transform_edt(excluded, return_distances=False, return_indices=True)\n            background = gray[tuple(indices)]\n            edit = excluded | (alpha > .005)\n            ink_samples = gray[gray < light - .35]\n            ink_level = float(np.median(ink_samples)) if ink_samples.size else 0.\n            composed = background*(1-alpha) + ink_level*alpha\n            target_crop = self.target[region]\n            target_crop[edit] = composed[edit, None]\n            if self.background is not None:\n                background_crop=self.background[region]\n                opacity=np.where(excluded, np.clip(1-gray/np.maximum(background, .01),0,1), 0)\n                self.source[region]=background_crop*(1-opacity[...,None])\n                target_crop[:]=self.source[region]\n                target_crop[edit]=(background_crop*(1-alpha[...,None]))[edit]\n            self.roi[region] = edit\n        if not self.roi.any():\n            raise ValueError(\'보정할 글자 영역이 없습니다.\')\n        self.weights = self.roi.astype(np.float32)\n        self._mask = torch.from_numpy(self.roi)[None, None]\n        self._target = torch.from_numpy(self.target).permute(2,0,1)[None]\n        self._source = torch.from_numpy(self.source).permute(2,0,1)[None]\n        self._background = (torch.from_numpy(self.background).permute(2,0,1)[None]\n                            if self.background is not None else None)\n\n    @torch.no_grad()\n    def project(self, rgb):\n        # The same silhouette sets both the ink to add and the ink to remove.\n        return torch.where(self._mask.to(rgb.device), self._target.to(rgb), rgb)\n\n    @torch.no_grad()\n    def diagnostics(self, rgb):\n        error = (rgb.mean(1, keepdim=True)-self._target.to(rgb).mean(1, keepdim=True))[self._mask.to(rgb.device)]\n        return dict(missing_ink=float(error.clamp_min(0).mean()),\n                    extra_ink=float((-error).clamp_min(0).mean()))\n\n    @torch.no_grad()\n    def finish(self, rgb):\n        background=self._background.to(rgb) if self._background is not None else None\n        return finish_ink_rgb(rgb,self._source.to(rgb),self._mask.to(rgb.device),background)\n', 'repair_audit/flattened_text_repair.py': '"""Repair a single flattened 2x2 image using local deletion inpainting."""\nfrom dataclasses import dataclass\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import binary_dilation, grey_closing\nimport torch\n\nfrom stroke_model.source_structure import SourceConfig\nfrom stroke_model.target_stroke_plan import TargetImageGuide, TargetPlanConfig, build_target_plan\nfrom .target_guided_repair import extract_sheet_masks\n\n\n@dataclass\nclass FlattenedPlan:\n    source: Image.Image\n    proxy: Image.Image\n    opacity: np.ndarray\n    aligned_alpha: np.ndarray\n    erase: np.ndarray\n    add: np.ndarray\n    report: dict\n\n\ndef white_background_proxy(source):\n    """Keep dark neutral ink and nearby antialiasing in the same pixel coordinates."""\n    rgb = np.asarray(source.convert(\'RGB\'), np.float32)/255\n    gray = rgb @ np.array([.299, .587, .114], np.float32)\n    chroma = np.ptp(rgb, axis=-1)\n    core = (gray <= .30) & (chroma <= .12)\n    edge = (gray <= .58) & (chroma <= .23) & binary_dilation(core, iterations=2)\n    window = max(9, round(source.width / 12) | 1)\n    light = grey_closing(gray, size=(window, window))\n    opacity = np.clip((light-gray)/np.maximum(light, .15), 0, 1)\n    opacity[~(core | edge)] = 0\n    proxy = Image.fromarray(np.rint((1-opacity)*255).astype(np.uint8)).convert(\'RGB\')\n    return proxy, opacity\n\n\ndef prepare_flattened_plan(source, conditions, extractor):\n    """Extract strokes from the white proxy, retaining the full-color image for repair."""\n    source = source.convert(\'RGB\')\n    if source.width != source.height or source.width % 16 or len(conditions) != 4:\n        raise ValueError(\'Use a square 2x2 sheet with four references and dimensions divisible by 16\')\n    proxy, opacity = white_background_proxy(source)\n    observed = extract_sheet_masks(proxy, conditions, extractor)\n    aligned, rendered = build_target_plan(proxy, conditions, observed,\n        registration_config=SourceConfig(rotation=0., shear=0.))\n    alpha = np.zeros_like(opacity)\n    ink = np.zeros_like(opacity, dtype=bool)\n    for glyph, target, masks in zip(aligned.report[\'glyphs\'], rendered, observed, strict=True):\n        x0,y0,x1,y1 = glyph[\'bbox\']\n        box = np.s_[y0:y1,x0:x1]\n        size = (x1-x0,y1-y0)\n        alpha[box] = np.asarray(Image.fromarray(target[\'aligned_strokes\'].max(0)).resize(size, Image.Resampling.BILINEAR))\n        support = np.asarray(Image.fromarray(masks.max(0)).resize(size, Image.Resampling.BILINEAR)) > .22\n        ink[box] = binary_dilation(support, iterations=3) & (opacity[box] > .12)\n    # Never open up a hole inside a target stroke; include a small edge margin for residual ink.\n    erase = binary_dilation(ink, iterations=2) & (alpha <= .03)\n    add = (alpha - opacity) > .06\n    report = dict(aligned.report, input_mode=\'single_flattened_image\',\n                  extraction_input=\'neutral_ink_on_white\',\n                  erase_pixels=int(erase.sum()), add_pixels=int(add.sum()))\n    return FlattenedPlan(source, proxy, opacity, alpha, erase, add, report)\n\n\ndef inpaint_deleted_ink(plan, pipe, *, steps=25, seed=2026):\n    if not plan.erase.any():\n        return plan.source.copy()\n    mask = Image.fromarray(plan.erase.astype(np.uint8)*255)\n    result = pipe(prompt=\'continuous background matching the surrounding color and texture\',\n        negative_prompt=\'text, glyphs, letters, lines of writing\', image=plan.source, mask_image=mask,\n        width=plan.source.width, height=plan.source.height, strength=1., num_inference_steps=steps,\n        guidance_scale=5., generator=torch.Generator(\'cpu\').manual_seed(seed))\n    if getattr(result,\'nsfw_content_detected\',None) and result.nsfw_content_detected[0]:\n        raise ValueError(\'인페인팅 결과가 필터링됐습니다.\')\n    patch = result.images[0].convert(\'RGB\')\n    if patch.size!=plan.source.size:\n        raise ValueError(\'Inpaint output must match the source dimensions\')\n    return Image.composite(patch, plan.source, mask)\n\n\nclass WhiteStrokeGuide(TargetImageGuide):\n    def __init__(self, plan):\n        source = np.asarray(plan.proxy, np.float32)/255\n        target = source.copy()\n        weights = (plan.erase | plan.add).astype(np.float32)\n        if not weights.any():\n            raise ValueError(\'추가하거나 지울 획 영역이 잡히지 않았습니다.\')\n        target[plan.erase] = 1\n        target[plan.add] = (1-plan.aligned_alpha[plan.add])[:,None]\n        super().__init__(source, target, weights, [], dict(plan.report,strokes=[]),\n                         TargetPlanConfig(maximum_step_blend=1.))\n        self.metadata.update(kind=\'white_stroke_layer\', outside_roi=\'original_proxy_pixels\',\n                             inside_roi=\'grayscale_guided_ink\', score_role=\'none\')\n\n    @torch.no_grad()\n    def finish(self, rgb):\n        source = torch.as_tensor(self.source,device=rgb.device).to(rgb).permute(2,0,1)[None]\n        mask = torch.as_tensor(self.roi,device=rgb.device)[None,None]\n        gray = rgb.mean(dim=1,keepdim=True).expand_as(rgb)\n        return torch.where(mask,gray,source)\n\n\ndef composite_repaired_ink(plan, repaired_white, filled):\n    """Transfer new ink opacity and the inpainted deletion patch onto the original RGB."""\n    if repaired_white.size!=plan.source.size or filled.size!=plan.source.size:\n        raise ValueError(\'Repair layers must match the source dimensions\')\n    source = np.asarray(plan.source, np.uint8)\n    finished = source.copy()\n    finished[plan.erase] = np.asarray(filled.convert(\'RGB\'))[plan.erase]\n    before = np.asarray(plan.proxy.convert(\'L\'), np.float32)/255\n    after = np.asarray(repaired_white.convert(\'L\'), np.float32)/255\n    # The input already has the old ink. Apply only the extra darkening.\n    transmission = np.clip(after/np.maximum(before, 1/255), 0, 1)\n    mask = plan.add & ~plan.erase\n    finished[mask] = np.rint(source[mask]*transmission[mask,None]).astype(np.uint8)\n    return Image.fromarray(finished)\n'}
SOURCE_HASHES={'stroke_model/reference_model.py': 'df3445a74e2543ee9e86037b03316650918231f99da41dcf1efb0fb81b7f597f', 'stroke_model/stroke_structure.py': '9d6b782def7fa36195db5d45410fd689ec130a14fdf626f23b6da402c767519f', 'stroke_model/source_structure.py': 'd695bc04705b63b5b6131bdb552ea5f31fc719eade43960ee01723277e177de5', 'stroke_model/target_stroke_plan.py': 'b2a92f4665654e7b7852acffd558d27b1193b6b209eff7d9675a863e6fc2cca6', 'diffusion_model/stroke_projected_diffusion.py': '34a100cf2da07f18581c224178db91e1c15b6bbd0a01d9c129309e78e266e883', 'repair_audit/stage1.py': '83a898997f8b75a68ffff25675a5903064465fe7ad3188e4a9a70d549488a94f', 'repair_audit/target_guided_repair.py': 'fbb06ab40bbe3bb3f7f8f24ccfc5cab2e62c35f28dc16118c946b9cc164513b5', 'repair_audit/fresh_target_inputs.py': '1ad3160189cfe7a370c8c3070b6d0f92eef96cf2216caff439be9234e2c01841', 'diffusion_model/pilot_data.py': '27bd13c4d6f8167adfe6256429224493cec87712fdb1058f5352c18db2d5fa32', 'diffusion_model/score_pilot.py': 'c1dbda0e184610b2b9d68b45720df97e7dfeeb3f4cbc147a1950b1cc2a1462ab', 'stroke_model/convert_reference_csv.py': 'b5d267f8a81e0cd88ddb479ce4910c331dd1873aaba68bb3b94688d5abcb10a6', 'stroke_model/prepare_stroke_zips.py': '58cc750d5ba2a41421dc32d2f44b11b6fc2f2c859427cb42ed3f6d338a10a939', 'stroke_model/reference_only_data.py': '7c84c148be9113d48321d237c94532ac09fe007ce48ed3b00bdb66661a1de48c', 'repair_audit/selectable_target_inputs.py': '3d5a55f0e652611238ea4ca2def0ac4119fe0daefd37f009129417c2659c5980', 'repair_audit/selection_cache.py': 'f5e24c5b77f53cb8cb914342d8e17984d16ebb9f4866be6626688966a0e5c9ab', 'stroke_model/ink_target_guide.py': '272a5740f9bc93f424449ea1669f813a389f5953e40a55fb7db420485e8f6cfc', 'repair_audit/flattened_text_repair.py': '936c48a579861c607510957818b25296da6e6ea882da635cf0da7440cf4a575d'}
RUNTIME = Path('/content/hsqm_target_guided_runtime')
for name, code in RUNTIME_HELPERS.items():
    if hashlib.sha256(code.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise ValueError('Embedded source hash mismatch: '+name)
    target = RUNTIME/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(code, encoding='utf-8')
    (target.parent/'__init__.py').touch()
# A rerun must not keep modules imported by an older notebook in this session.
import importlib
for module_name in list(sys.modules):
    if module_name.split('.')[0] in ('repair_audit', 'stroke_model', 'diffusion_model'):
        del sys.modules[module_name]
if str(RUNTIME) in sys.path: sys.path.remove(str(RUNTIME))
sys.path.insert(0, str(RUNTIME))
importlib.invalidate_caches()
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import drive, files
from repair_audit.stage1 import load_conditions, file_hash, json_write
from repair_audit.target_guided_repair import repair_generated_sheet, extract_sheet_masks
from stroke_model.target_stroke_plan import TargetPlanConfig
from repair_audit.fresh_target_inputs import generate_fresh_cases, canonical_sheet
print('실행 장치:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (전처리·선택 가능, 모델 실행은 GPU 필요)')
from google.colab import output
output.enable_custom_widget_manager()
import ipywidgets as widgets
import numpy as np
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.score_pilot import training_catalog, load_catalog_reference
from repair_audit.selectable_target_inputs import (prepare_selected_sheet, save_prototype,
    load_prototype, load_reference_image)
from repair_audit.selection_cache import prepare_selection_cache,selection_thumbnails,thumbnail,reference_identity
from io import BytesIO
from PIL import ImageOps
from repair_audit.flattened_text_repair import prepare_flattened_plan, inpaint_deleted_ink, WhiteStrokeGuide, composite_repaired_ink
from diffusion_model.stroke_projected_diffusion import run_projected_pair


In [ ]:
# 2. Drive의 기존 체크포인트·기준 자료 준비 (결과 ZIP 불필요)
CHECKPOINT = '/content/drive/Shareddrives/2026 자율연구/HSQM/v2_topology_runs/20261104_214705_208224/best.pt'
DATA_ROOT = '/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/automated_train_dataset'
CSV_PATH = None  # 기본: best.pt 옆 selected_paths.csv
TRAIN_MANIFEST = None  # 현재 세션의 유효한 manifest가 있으면 지정 가능
CANVAS_SIZE = 576
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION = '451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
if not Path(CHECKPOINT).is_file(): raise FileNotFoundError('CHECKPOINT 경로를 확인하세요.')
PREPARATION = prepare_pilot_data(CHECKPOINT,DATA_ROOT,csv_path=CSV_PATH,manifest=TRAIN_MANIFEST,
    local_root='/content/hsqm_selectable_reference_data',reference_only=True)
TRAIN_MANIFEST = Path(PREPARATION['manifest'])
CATALOG = training_catalog(TRAIN_MANIFEST)
if not CATALOG: raise ValueError('선택 가능한 기준 글자가 없습니다.')
SOURCE_CATALOG = CATALOG
print('기준 목록:',len(CATALOG),'개. 3번 전처리를 실행하세요.')


In [ ]:
# 3. 선택용 전처리. 원본 읽기·로컬 자료 복사·썸네일 생성은 여기서 한 번 수행
def preparation_progress(done,total):
    if done==1 or done%10==0 or done==total: print('선택 자료 준비',done,'/',total,flush=True)
CATALOG = prepare_selection_cache(SOURCE_CATALOG,'/content/hsqm_picker_cache',
    source_fingerprint=file_hash(TRAIN_MANIFEST),progress=preparation_progress)
SELECTION_THUMBNAILS = selection_thumbnails(CATALOG)
print('기준 목록 준비 완료. 4번에서 목표 글자를 고르세요.')


In [ ]:
# 4. 각 칸의 목표 글자와 기준 획 선택
entries_by_label = {f"{e.get('resolved_character') or '?'} | row {e['manifest_row']} | {(e.get('reference') or '').rsplit('/',1)[-1]}":e for e in CATALOG}
labels = list(entries_by_label)
choices = [widgets.Combobox(options=labels,value=labels[min(i,len(labels)-1)],
    ensure_option=True,description=name,continuous_update=False,layout=widgets.Layout(width='90%'))
    for i,name in enumerate(['왼쪽 위','오른쪽 위','왼쪽 아래','오른쪽 아래'])]
preview = widgets.Output()
def show_reference(change=None):
    with preview:
        preview.clear_output(wait=True)
        if any(choice.value not in entries_by_label for choice in choices):return
        entries = [entries_by_label[choice.value] for choice in choices]
        display(canonical_sheet([dict(reference=load_reference_image(entry)) for entry in entries],256))
for choice in choices:choice.observe(show_reference,names='value')
display(widgets.VBox([*choices,preview]));show_reference()


In [ ]:
# 5. 배경과 글자가 합쳐진 이미지 업로드
source = None
uploaded = files.upload()
if len(uploaded)!=1:raise ValueError('정사각형 2×2 입력 이미지 한 장을 올리세요.')
source_name, contents = next(iter(uploaded.items()))
with Image.open(BytesIO(contents)) as image:
    image = ImageOps.exif_transpose(image).convert('RGB')
    if image.width != image.height:raise ValueError('2×2 시트는 정사각형이어야 합니다.')
    source = image.resize((CANVAS_SIZE,CANVAS_SIZE),Image.Resampling.LANCZOS)
display(source.resize((320,320)))


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('모델 실행에는 GPU 런타임이 필요합니다. 전처리·선택은 CPU에서도 가능합니다.')
# 6. 기존 획 추출기와 SD1.5 로딩 (추가 학습 없음)
from stroke_model.reference_model import ReferenceStrokeModel
from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
if state.get('architecture') != 'reference_v2_topology_loss_v1':
    raise ValueError('reference_v2_topology_loss_v1 체크포인트가 필요합니다.')
extractor = ReferenceStrokeModel(image_size=state['image_size'], width=state['width'], pretrained=False)
extractor.load_state_dict(state['model'], strict=True)
EXTRACTOR_SIZE = state['image_size']
extractor.eval().requires_grad_(False)  # 추출할 때만 GPU에 올림
del state; gc.collect(); torch.cuda.empty_cache()
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(MODEL_ID, revision=REVISION,
    torch_dtype=torch.float16, use_safetensors=True)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, clip_sample=False, thresholding=False)
pipe.to('cuda'); pipe.enable_attention_slicing(); pipe.enable_vae_slicing()
for module in (pipe.unet, pipe.vae, pipe.text_encoder): module.eval().requires_grad_(False)
pipe.vae.to(dtype=torch.float32)
SCHEDULER_CONFIG = dict(pipe.scheduler.config)
import importlib.metadata
VERSIONS = {name:importlib.metadata.version(name) for name in
    ('torch','diffusers','transformers','timm','numpy','scipy','scikit-image')}
print('로딩 완료. 7번에서 마스크를 확인하세요.')


In [ ]:
# 7. 획 추출 및 삭제·추가 마스크 확인
plan = None
if source is None:raise ValueError('5번에서 입력을 올리세요.')
entries = [entries_by_label[choice.value] for choice in choices]
planned_choices = [choice.value for choice in choices]
conditions = [dict(zip(('reference','strokes'),load_catalog_reference(entry))) for entry in entries]
if any(c['reference'].shape[-2:]!=(EXTRACTOR_SIZE,EXTRACTOR_SIZE) for c in conditions):
    raise ValueError('기준 이미지 해상도가 추출기와 다릅니다.')
try:
    extractor.to('cuda')
    plan = prepare_flattened_plan(source,conditions,extractor)
finally:
    extractor.to('cpu');torch.cuda.empty_cache()
overlay = np.asarray(source).copy()
overlay[plan.erase] = [255,64,64]
overlay[plan.add] = [75,155,255]
panel = Image.new('RGB',(960,320))
for index,image in enumerate([source,plan.proxy,Image.fromarray(overlay)]):
    panel.paste(image.resize((320,320)),(index*320,0))
print('왼쪽 원본 / 가운데 획 추출용 흰 배경 / 오른쪽 삭제(빨강)·추가(파랑)')
display(panel)


In [ ]:
# 8. 흰 배경 획 보정 → 삭제 자리 인페인팅 → 원본 배경 합성
from diffusers import AutoPipelineForInpainting
if plan is None:raise ValueError('7번에서 마스크를 먼저 확인하세요.')
if source is None or planned_choices!=[choice.value for choice in choices] or not np.array_equal(np.asarray(plan.source),np.asarray(source)):
    raise ValueError('입력이나 기준이 바뀌었습니다. 7번에서 마스크를 다시 만드세요.')
OUTPUT = Path('/content')/('flattened_text_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
OUTPUT.mkdir()
source.save(OUTPUT/'input.png');plan.proxy.save(OUTPUT/'ink_proxy.png')
Image.fromarray(plan.erase.astype(np.uint8)*255).save(OUTPUT/'erase_mask.png')
Image.fromarray(plan.add.astype(np.uint8)*255).save(OUTPUT/'add_mask.png')
json_write(OUTPUT/'run.json',dict(plan.report,source_name=source_name,reference_rows=[e['manifest_row'] for e in entries],
    checkpoint_sha256=file_hash(CHECKPOINT),source_hashes=SOURCE_HASHES,versions=VERSIONS,model_id=MODEL_ID,revision=REVISION))
guide = WhiteStrokeGuide(plan)
target = Image.fromarray(np.rint(guide.target*255).astype(np.uint8))
target.save(OUTPUT/'target_white.png')
report = run_projected_pair(pipe,plan.proxy,guide,OUTPUT/'reverse',steps=60,strength=.25,
    prompt='black Korean typography on a plain white background',guided_only=True)
with Image.open(OUTPUT/'reverse/stroke_projected.png') as image:
    repaired_white = image.convert('RGB')
repaired_white.save(OUTPUT/'repaired_white.png')
try:
    pipe.vae.to(dtype=pipe.unet.dtype)
    inpaint = AutoPipelineForInpainting.from_pipe(pipe)
    inpaint.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    filled = inpaint_deleted_ink(plan,inpaint,steps=25)
finally:
    pipe.vae.to(dtype=torch.float32)
filled.save(OUTPUT/'inpainted.png')
result = composite_repaired_ink(plan,repaired_white,filled)
result.save(OUTPUT/'final.png')
panel = Image.new('RGB',(960,240))
for index,image in enumerate([source,repaired_white,filled,result]):
    panel.paste(image.resize((240,240)),(index*240,0))
panel.save(OUTPUT/'comparison.png');display(panel)
print('왼쪽부터 컬러 입력 / 흰 배경 획 보정 / 삭제 자리를 메운 원본 / 컬러 최종 결과(final.png)')


In [ ]:
# 9. 결과 ZIP 다운로드
files.download(shutil.make_archive(str(OUTPUT),'zip',OUTPUT))
